# 🎯 Resume Optimizer - QLoRA Fine-tuned LLM

> **Project Overview:** This notebook demonstrates an end-to-end pipeline for building a resume optimization system using QLoRA fine-tuning on the Qwen3 model.

## 📋 Table of Contents

1. **Setup & Dependencies** - Install required packages and import libraries
2. **Dataset Creation** - Extract text from 1800+ resumes (PDF, DOCX, DOC)
3. **Job Scraper** - Scrape job descriptions from job posting websites
4. **Data Cleaning & Combination** - Merge resume data with job descriptions
5. **Ollama Resume Generator** - Generate tailored resumes using local Ollama model
6. **Gemini Batch API** - Process resumes at scale using Google's Gemini API
7. **Training Dataset Preparation** - Format data for fine-tuning
8. **Base Model Testing** - Test the base Qwen3 model before fine-tuning
9. **LoRA Fine-tuning** - Train the model using QLoRA technique
10. **Inference** - Generate tailored resumes with the fine-tuned model

---

## 1. 📦 Setup & Dependencies

### 1.1 Installation Commands (Optional)
Run the cells below if packages are not already installed. These are commented out by default.

In [ ]:
# Optional: Install Google Chrome for web scraping (Linux only)
# !wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# !apt-get install -y ./google-chrome-stable_current_amd64.deb

In [ ]:
# Optional: Install web scraping dependencies
# !pip install selenium webdriver-manager beautifulsoup4

  Using cached selenium-4.38.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio-0.32.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
Using cached selenium-4.38.0-py3-none-any.whl (9.7 MB)
Using cached trio-0.32.0-py3-none-any.whl (512 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cach

In [ ]:
# Optional: Install all project dependencies
# !pip install pandas numpy tqdm requests PyPDF2 python-docx selenium webdriver-manager \
#              beautifulsoup4 lxml pywin32 pyarrow google-genai torch transformers \
#              datasets trl peft bitsandbytes accelerate

### 1.2 Import Libraries

Import all required libraries for data processing, web scraping, ML/AI, and file handling.

In [ ]:
# Standard library imports
import os
import json
import re
import time
import logging
import base64
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, Any, Tuple, List

# Data processing
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests

# Document parsing
from PyPDF2 import PdfReader
from PyPDF2.errors import PdfReadError
from docx import Document
from docx.opc.constants import RELATIONSHIP_TYPE as RT
from docx.opc.exceptions import PackageNotFoundError

# Web scraping
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# Google AI
from google import genai
from google.genai import types

# PyTorch and Transformers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, PeftModel
import importlib.util

print("✅ All libraries imported successfully!")

c:\ProgramData\anaconda3\envs\finetune\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 2. 📄 Dataset Creation from 1800+ Resumes

This section extracts text content from resume documents in multiple formats (PDF, DOCX, DOC).

### Key Features:
- **PDF Extraction:** Uses PyPDF2 to extract text and detect embedded images
- **DOCX Extraction:** Parses Word documents and identifies image relationships  
- **DOC Extraction:** Uses Windows COM automation (requires pywin32 and Microsoft Word)
- **Output Format:** JSONL with filename, filetype, resume text, and image detection flag

### Configuration

In [ ]:
# Windows-specific imports for legacy .doc file support
try:
    import win32com.client as win32
    import pywintypes
    print("✅ pywin32 available - .doc file support enabled")
except ImportError:
    win32 = None
    pywintypes = None
    print("⚠️ pywin32 not available - .doc files will be skipped")

# Define recoverable errors that shouldn't stop processing
RECOVERABLE_EXTRACTION_ERRORS = (
    ValueError,
    OSError,
    PdfReadError,
    PackageNotFoundError,
    KeyError,
    RuntimeError,
)
# if pywintypes is not None and hasattr(pywintypes, "com_error"):
#     RECOVERABLE_EXTRACTION_ERRORS = RECOVERABLE_EXTRACTION_ERRORS + (
#         pywintypes.com_error,  # type: ignore[attr-defined]
#     )

In [ ]:
# Configuration: File paths and supported formats
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".doc"}
DEFAULT_INPUT_PATH = Path(r"C:\Users\Harsha\Documents\Repo\Colab-proj-2\data\resumes")
DEFAULT_OUTPUT_PATH = Path("files\dataset\extracted-resume-data\resume_text.jsonl")

print(f"📁 Input path: {DEFAULT_INPUT_PATH}")
print(f"📁 Output path: {DEFAULT_OUTPUT_PATH}")
print(f"📄 Supported formats: {SUPPORTED_EXTENSIONS}")

### 2.1 Document Extraction Functions

The following code defines helper classes and functions for extracting text from various document formats.

In [ ]:
class WordAutomationClient:
    """Thin wrapper around Word COM automation to read legacy .doc files."""

    def __init__(self) -> None:
        if win32 is None:
            raise ImportError("pywin32 is required for .doc support on Windows")
        self._word = win32.Dispatch("Word.Application")
        self._word.Visible = False

    def close(self) -> None:
        if self._word is not None:
            self._word.Quit()
            self._word = None

    def __enter__(self) -> "WordAutomationClient":
        return self

    def __exit__(self, exc_type, exc, exc_tb) -> None:  # type: ignore[override]
        self.close()

    def extract_doc(self, file_path: Path) -> tuple[str, bool]:
        if self._word is None:
            raise RuntimeError("Word automation client is closed")
        document = self._word.Documents.Open(str(file_path))
        try:
            text = document.Content.Text
            contains_images = bool(document.InlineShapes.Count or document.Shapes.Count)
        finally:
            document.Close(False)
        return text.strip(), contains_images


def extract_text_from_pdf(file_path: Path) -> str:
    """Extract text from a PDF file."""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text.strip()


def extract_text_from_docx(file_path: Path) -> str:
    """Extract text from a DOCX file."""
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        text += para.text + "\n"
    return text.strip()


def extract_text(file_path: Path) -> str:
    """Extract text from a document file based on its extension."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(file_path)
    if suffix == ".docx":
        return extract_text_from_docx(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def _xobject_dict_contains_images(x_objects) -> bool:
    if not x_objects:
        return False
    # Dereference x_objects if it's an indirect object
    try:
        x_objects = x_objects.get_object()
    except AttributeError:
        pass
    if not hasattr(x_objects, "values"):
        return False
    for obj in x_objects.values():
        try:
            x_obj = obj.get_object()
        except AttributeError:
            x_obj = obj
        subtype = x_obj.get("/Subtype")
        if subtype == "/Image":
            return True
        if subtype == "/Form":
            child_resources = x_obj.get("/Resources")
            child_x_objects = None
            if child_resources:
                # Dereference indirect objects
                try:
                    child_resources = child_resources.get_object()
                except AttributeError:
                    pass
                if hasattr(child_resources, "get"):
                    child_x_objects = child_resources.get("/XObject")
            if _xobject_dict_contains_images(child_x_objects):
                return True
    return False


def pdf_has_images(file_path: Path) -> bool:
    reader = PdfReader(file_path)
    for page in reader.pages:
        if hasattr(page, "images") and page.images:
            return True
        resources = page.get("/Resources")
        if not resources:
            continue
        # Dereference indirect objects
        try:
            resources = resources.get_object()
        except AttributeError:
            pass
        if not hasattr(resources, "get"):
            continue
        x_objects = resources.get("/XObject")
        if _xobject_dict_contains_images(x_objects):
            return True
    return False


def docx_has_images(file_path: Path) -> bool:
    doc = Document(file_path)
    return any(rel.reltype == RT.IMAGE for rel in doc.part.rels.values())


def has_images(file_path: Path) -> bool:
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return pdf_has_images(file_path)
    if suffix == ".docx":
        return docx_has_images(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def collect_documents(target: Path) -> List[Path]:
    if target.is_file():
        return [target]
    if target.is_dir():
        return sorted(
            [f for f in target.rglob("*") if f.suffix.lower() in SUPPORTED_EXTENSIONS]
        )
    raise FileNotFoundError(f"'{target}' is not a valid file or directory.")


def process_document(
    file_path: Path, word_client: Optional["WordAutomationClient"]
) -> Dict[str, str | bool]:
    suffix = file_path.suffix.lower()
    if suffix == ".doc":
        if word_client is None:
            raise RuntimeError(
                ".doc support requires Microsoft Word and pywin32; both appear unavailable."
            )
        resume_text, contains_images = word_client.extract_doc(file_path)
    else:
        resume_text = extract_text(file_path)
        contains_images = has_images(file_path)

    return {
        "filename": file_path.name,
        "filetype": suffix,
        "resume_text": resume_text,
        "has_images": contains_images,
    }


def main() -> None:
    # Update these paths to control which resumes are processed and where JSONL output lands.
    input_path = DEFAULT_INPUT_PATH
    output_path = DEFAULT_OUTPUT_PATH

    try:
        documents = collect_documents(input_path)
    except FileNotFoundError as exc:
        print(exc)
        return

    if not documents:
        print(f"No PDF/DOC/DOCX files found under '{input_path}'.")
        return

    needs_word = any(path.suffix.lower() == ".doc" for path in documents)
    word_client: Optional[WordAutomationClient] = None
    if needs_word:
        try:
            word_client = WordAutomationClient()
        except ImportError as exc:
            print(f"Cannot open .doc files: {exc}")
            return

    processed_count = 0
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with output_path.open("w", encoding="utf-8") as fh:
            for doc_path in documents:
                try:
                    record = process_document(doc_path, word_client)
                except RECOVERABLE_EXTRACTION_ERRORS as exc:
                    print(f"Skipping {doc_path}: {exc}")
                    continue
                fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                processed_count += 1
                print(f"Processed {doc_path.name}")
    finally:
        if word_client is not None:
            word_client.close()

    if processed_count:
        print(
            f"Wrote metadata for {processed_count} file(s) to '{output_path.as_posix()}'"
        )
    else:
        print("No files were successfully processed; see logs above for details.")



In [ ]:
# Execute the resume extraction pipeline
if __name__ == "__main__":
    main()

---

## 3. 🔍 Job Scraper

This section scrapes job descriptions from job posting websites using Selenium WebDriver.

### Features:
- **Headless Browser:** Runs Chrome in headless mode for efficiency
- **Rate Limiting:** Configurable delays between requests to avoid blocking
- **Progress Tracking:** Real-time progress bar with success/failure counts
- **Incremental Saving:** Results saved after each scrape to prevent data loss

### 3.1 Network Check
First, verify internet connectivity by checking the external IP address.

In [ ]:
# Verify internet connectivity
try:
    response = requests.get('https://api64.ipify.org?format=json', timeout=10)
    response.raise_for_status()
    ip_address = response.json().get('ip')
    print(f"✅ Connected! Your IP Address: {ip_address}")
except requests.exceptions.RequestException as e:
    print(f"❌ Network error: {e}")

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
class JobScraper:
    """Handles web scraping of job descriptions from various job posting websites."""

    def __init__(self, headless=True, wait_time=5, delay=2):
        """
        Initialize the job description scraper.

        Args:
            headless (bool): Run browser in headless mode
            wait_time (int): Time to wait for page to load (seconds)
            delay (int): Delay between requests (seconds)
        """
        self.headless = headless
        self.wait_time = wait_time
        self.delay = delay
        self.driver = None

    def setup_driver(self):
        """Set up Chrome WebDriver with appropriate options."""
        options = webdriver.ChromeOptions()
        if self.headless:
            options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')

        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )
        return self.driver

    def close_driver(self):
        """Close the WebDriver if it exists."""
        if self.driver:
            self.driver.quit()
            self.driver = None

    def scrape_job(self, job_url):
        """
        Extract job description from a job posting URL.

        Args:
            job_url (str): URL of the job posting

        Returns:
            tuple: (job_description, status)
        """
        if not self.driver:
            self.setup_driver()

        try:
            self.driver.get(job_url)
            time.sleep(self.wait_time)

            soup = BeautifulSoup(self.driver.page_source, 'html.parser')

            body_text = soup.body.get_text() if soup.body else ""

            # Extract the job description part
            # Assuming it starts after job details like Full-time, Onsite, etc.
            lines = body_text.split('\n')
            desc_start = False
            description = []

            for line in lines:
                line = line.strip()
                if any(keyword in line for keyword in ['Full-time', 'Onsite', 'Remote', 'Hybrid', 'Part-time']):
                    desc_start = True
                if desc_start and line:
                    description.append(line)

            job_text = ' '.join(description)

            if len(job_text) < 100:
                return None, "Description too short (< 100 chars)"

            # Truncate very long descriptions
            if len(job_text) > 10000:
                job_text = job_text[:10000] + "..."

            return job_text, "Success"

        except Exception as e:
            logger.error(f"Error scraping URL {job_url}: {str(e)}")
            return None, f"Error: {str(e)}"

    def scrape_from_csv(self, input_csv, output_csv=None):
        """Scrape jobs from CSV file with URLs."""

        if not os.path.exists(input_csv):
            logger.error(f"Input CSV file not found: {input_csv}")
            return None

        # Read input CSV
        try:
            df = pd.read_csv(input_csv)
        except Exception as e:
            logger.error(f"Error reading CSV: {e}")
            return None

        # Find URL column - look for 'Apply', 'url', or 'link' columns
        url_column = None

        # Priority order: Apply > URL > Link
        column_priorities = ['apply', 'url', 'link']

        for priority_col in column_priorities:
            for col in df.columns:
                if priority_col in col.lower():
                    url_column = col
                    break
            if url_column:
                break

        if url_column is None:
            logger.error("No URL column found in CSV. Expected column with 'Apply', 'URL', or 'Link' in name.")
            logger.info(f"Available columns: {list(df.columns)}")
            return None

        logger.info(f"Found {len(df)} URLs to scrape in column '{url_column}'")

        # Create output filename if not provided
        if output_csv is None:
            timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
            output_csv = f"data/scraped_jobs_{timestamp}.csv"
            os.makedirs("data", exist_ok=True)

        # Create CSV file with headers if it doesn't exist
        file_exists = os.path.exists(output_csv)
        if not file_exists:
            with open(output_csv, 'w', encoding='utf-8', newline='') as f:
                f.write('URL,Job Description,Scrape Status\n')

        # Scrape each URL
        successful = 0
        failed = 0

        try:
            with tqdm(total=len(df), desc="Scraping jobs") as pbar:
                for idx, row in df.iterrows():
                    job_url = row[url_column]

                    if pd.isna(job_url) or not job_url.strip():
                        # Save immediately to CSV
                        result_df = pd.DataFrame([[job_url, "", "Empty URL"]],
                                                columns=['URL', 'Job Description', 'Scrape Status'])
                        result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')
                        failed += 1
                        pbar.update(1)
                        continue

                    # Scrape job
                    description, status = self.scrape_job(job_url)

                    # Save immediately to CSV after each scrape
                    result_df = pd.DataFrame([[
                        job_url,
                        description if description else "",
                        status
                    ]], columns=['URL', 'Job Description', 'Scrape Status'])

                    result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')

                    if status == "Success":
                        successful += 1
                    else:
                        failed += 1

                    pbar.set_postfix({
                        'Success': successful,
                        'Failed': failed,
                        'Rate': f"{successful/(successful+failed)*100:.1f}%" if (successful+failed) > 0 else "0%"
                    })

                    # Delay between requests
                    time.sleep(self.delay)
                    pbar.update(1)

            logger.info(f"Results saved to {output_csv}")
            logger.info(f"Summary: {successful} successful, {failed} failed")
            return output_csv,result_df

        except Exception as e:
            logger.error(f"Error during scraping: {e}")
            logger.info(f"Partial results saved to {output_csv}")
            logger.info(f"Summary before error: {successful} successful, {failed} failed")
            return output_csv,result_df
        finally:
            self.close_driver()

---

## 4. 🧹 Data Cleaning & Combination

This section processes and integrates the scraped job descriptions with the resume dataset.

### Pipeline Steps:
1. **Load Data:** Read scraped job data (CSV) and resume text data (JSONL)
2. **Filter Resumes:** Remove entries containing images (can't extract text reliably)
3. **Clean Text:** Remove non-ASCII characters and normalize whitespace
4. **Match Data:** Randomly assign job descriptions to resumes for training pairs
5. **Export:** Save combined dataset in multiple formats (CSV, Parquet, JSONL)

### 4.1 Configure Paths

In [ ]:
# Configuration for job scraping
input_csv = "files/job-scraper/job_links_merged.csv"
output_folder = 'files/job-scraper'

print(f"📁 Input CSV: {input_csv}")
print(f"📁 Output folder: {output_folder}")

In [ ]:
# Run the job scraping pipeline
os.makedirs(output_folder, exist_ok=True)

timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
output_csv = os.path.join(output_folder, f"scraped_jobs_{timestamp}.csv")

# Initialize scraper and run
scraper = JobScraper(headless=True, wait_time=5, delay=2)
print("🚀 Starting job scraping...")

result_file = scraper.scrape_from_csv(input_csv, output_csv)

if result_file:
    print(f"✅ Scraping complete! Results saved to: {result_file}")
else:
    print("❌ Scraping failed!")

### 4.2 Load and Preview Data

In [ ]:
# Load cleaned job data and resume text data
df_jobs = pd.read_csv('files/job-scraper/cleaned_scraped_jobs_21390711212025.csv')
df_resumes = pd.read_json('files/dataset/extracted-resume-data/resume_text.jsonl', lines=True)

print(f"📊 Jobs loaded: {len(df_jobs)} records")
print(f"📊 Resumes loaded: {len(df_resumes)} records")

### 4.3 Clean Resume Data

Remove resumes with images and clean text by removing non-ASCII characters and extra whitespace.

In [ ]:
# Filter out resumes containing images
df_resumes = df_resumes[~(df_resumes['has_images'] == True)]
print(f"📊 Resumes after removing images: {len(df_resumes)}")

def clean_text(text: str) -> str:
    """Remove non-ASCII characters and normalize whitespace."""
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII
    text = re.sub(r'\s+', ' ', text).strip()    # Normalize whitespace
    return text

# Apply cleaning to resume text and filename
df_resumes['resume_text'] = df_resumes['resume_text'].apply(clean_text)
df_resumes['filename'] = df_resumes['filename'].apply(clean_text)

# Filter out very short resumes
df_resumes = df_resumes[df_resumes['resume_text'].str.len() > 10]
print(f"📊 Resumes after length filter: {len(df_resumes)}")

# Additional cleaning: remove special characters
df_resumes['resume_text'] = (df_resumes['resume_text']
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

print(f"✅ Resume data cleaned successfully!")

In [ ]:
# Preview the cleaned data
print("=" * 50)
print("RESUMES DATAFRAME INFO:")
print("=" * 50)
df_resumes.info()
print("\n" + "=" * 50)
print("JOBS DATAFRAME INFO:")
print("=" * 50)
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB


### 4.4 Match Resumes with Job Descriptions

Randomly assign job descriptions to resumes to create training pairs for the model.

In [ ]:
# Create balanced job-resume pairs
job_descriptions = df_jobs['Job Description'].values
n_resumes = len(df_resumes)
n_jobs = len(job_descriptions)

# Evenly distribute job descriptions across resumes
repeats = (n_resumes // n_jobs) + 1
job_indices = np.tile(np.arange(n_jobs), repeats)[:n_resumes]

# Shuffle for randomization
np.random.seed(42)  # For reproducibility
np.random.shuffle(job_indices)

# Assign job descriptions
df_resumes['job_description'] = job_descriptions[job_indices]

# Display distribution statistics
print(f"📊 Total resumes: {len(df_resumes)}")
print(f"📊 Unique job descriptions: {df_resumes['job_description'].nunique()}")
print(f"\n📈 Distribution Statistics:")
print(f"   Min count per job: {df_resumes['job_description'].value_counts().min()}")
print(f"   Max count per job: {df_resumes['job_description'].value_counts().max()}")
print(f"   Mean count: {df_resumes['job_description'].value_counts().mean():.2f}")

Total resumes: 1565
Unique job descriptions: 245

Distribution of job descriptions (counts):
count    245.000000
mean       6.387755
std        0.882812
min        6.000000
25%        6.000000
50%        6.000000
75%        7.000000
max       13.000000
Name: count, dtype: float64

Min count: 6
Max count: 13

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
Index: 1565 entries, 0 to 2126
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1565 non-null   object
 1   filetype         1565 non-null   object
 2   resume_text      1565 non-null   object
 3   has_images       1565 non-null   bool  
 4   job_description  1565 non-null   object
dtypes: bool(1), object(4)
memory usage: 62.7+ KB


### 4.5 Export Combined Dataset

Save the combined dataset in multiple formats for flexibility.

In [ ]:
# Export to multiple formats
DATA_DIR = "./files/dataset/resume-data-with-job-description"
os.makedirs(DATA_DIR, exist_ok=True)

# Save in different formats
df_resumes.to_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df_resumes.to_json(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl'), orient='records', lines=True)

print("✅ Dataset exported to:")
print(f"   📄 {DATA_DIR}/final_resume_dataset.parquet")
print(f"   📄 {DATA_DIR}/final_resume_dataset.jsonl")

In [ ]:
# Verify exported files by comparing formats
DATA_DIR = "./files/dataset/resume-data-with-job-description"

# Read all three formats

df_parquet = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'))
df_jsonl = pd.read_json(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl'), lines=True)

# Compare shapes
print("=" * 50)
print("FORMAT COMPARISON")
print("=" * 50)
print(f"\n📊 Shape Comparison:")
print(f"   Parquet: {df_parquet.shape}")
print(f"   JSONL:   {df_jsonl.shape}")

# Compare file sizes
csv_size = os.path.getsize(os.path.join(DATA_DIR, 'final_resume_dataset.csv')) / (1024*1024)
parquet_size = os.path.getsize(os.path.join(DATA_DIR, 'final_resume_dataset.parquet')) / (1024*1024)
jsonl_size = os.path.getsize(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl')) / (1024*1024)

print(f"\n💾 File Size Comparison:")
print(f"   CSV:     {csv_size:.2f} MB")
print(f"   Parquet: {parquet_size:.2f} MB (most compact)")
print(f"   JSONL:   {jsonl_size:.2f} MB")


=== Shape Comparison ===
CSV:     (1565, 5)
Parquet: (1565, 5)
JSONL:   (1565, 5)

=== Columns Comparison ===
CSV columns:     ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
Parquet columns: ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
JSONL columns:   ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']

=== Data Types Comparison ===
CSV dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

Parquet dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

JSONL dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

=== File Size Comparison ===
CSV:     25.36 MB
Parquet: 9.48 MB
JSONL:   25.50 MB

=== Data Equality Check ===
C

,filename,filetype,resume_text,has_images,job_description
0,00_Willie_Ellis_Go_Python.docx,.docx,Willie Ellis Senior Software Engineer Buffalo ...,False,Hanger/Textiles jobs in United StatesOverviewC...
1,10272022 Resume.docx,.docx,SUMMARY Leverage my skills education and exper...,False,Production Associate - Garment Hanger/Inspecto...


In [ ]:
# Load the parquet file for subsequent processing (most efficient format)
DATA_DIR = "./files/dataset/resume-data-with-job-description"
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"📊 Loaded {len(df)} records from parquet file")

---

## 5. 🤖 Ollama Resume Generator (Local LLM)

This section uses a **local Ollama model** (Qwen3:8b) to generate tailored resumes based on resume text and job descriptions.

### Features:
- **Local Processing:** No API costs, data stays on your machine
- **Thinking Mode Support:** Works with models that use `<think>` tags
- **JSON Extraction:** Robust parsing of JSON from model responses
- **Progress Tracking:** Automatic saving after each generation

### 5.1 Define Resume Schema and Prompt Template

In [ ]:
# JSON Schema for tailored resume output (defines the expected structure)
RESUME_SCHEMA = '''{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"type":"string"},"location":{"type":"string"},"socials":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]}},"required":["name","email","phone","location"]},"summary":{"type":"string"},"experiences":{"type":"array","items":[{"type":"object","properties":{"designation":{"type":"string"},"companyName":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"points":{"type":"array","items":[{"type":"string"}]}},"required":["designation","companyName","location","start_date"]}]},"education":{"type":"array","items":[{"type":"object","properties":{"institution":{"type":"string"},"degree":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"gpa":{"type":"string"}},"required":["institution","degree","location","start_date","gpa"]}]},"skills":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"data":{"type":"array","items":[{"type":"string"}]}},"required":["name","data"]}]},"projects":{"type":"array","items":[{"type":"object","properties":{"projectName":{"type":"string"},"caption":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"url":{"type":"string"},"projectDetails":{"type":"array","items":[{"type":"string"}]},"externalSources":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]},"technologiesUsed":{"type":"array","items":[{"type":"string"}]}},"required":["projectName","location","projectDetails"]}]},"certifications":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"issuing_organization":{"type":"string"},"issue_date":{"type":"string"},"expiration_date":{"type":"string"},"credential_id":{"type":"string"},"url":{"type":"string"}},"required":["name","issuing_organization","issue_date","expiration_date","credential_id","url"]}]},"awards":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"extracurricular_achievements":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"languages":{"type":"array","items":[{"type":"object","properties":{"language":{"type":"string"},"proficiency":{"type":"string"}},"required":["language","proficiency"]}]}},"required":["personal_information","education","skills","extracurricular_achievements"]}'''

# Load prompt template from file
with open('dataset/prompt.txt', 'r', encoding='utf-8') as f:
    PROMPT_TEMPLATE = f.read()

print("✅ Schema and prompt template loaded!")
print(f"\n📄 Schema preview: {RESUME_SCHEMA[:100]}...")
print(f"\n📝 Prompt preview: {PROMPT_TEMPLATE[:200]}...")

Prompt template loaded successfully
Schema defined:
{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"ty...
Prompt template preview:
You create a tailored resume based on the job description.

Your task:
1. Read the RESUME_TEXT.
2. Read the JOB_DESCRIPTION.
3. Use only the information inside these two.
4. Follow the SCHEMA exactly.
5. Write a tailored resume in JSON using the SCHEMA.
6. Do not output anything outside the JSON.
7. If a field is missing in the resume, write a short, safe placeholder that fits the job.


RESUME_TEXT:
"""
<<PASTE RESUME HERE>>
"""

JOB_DESCRIPTION:
"""
<<PASTE JD HERE>>
"""

SCHEMA:
"""
<<PASTE J...


### 5.2 Ollama Resume Generator Class

This class handles communication with the local Ollama API and processes responses.

In [ ]:


class OllamaResumeGenerator:
    """Class to generate tailored resumes using local Ollama model (supports thinking models)."""
    
    def __init__(
        self,
        model_name: str = "qwen3:8b",
        base_url: str = "http://localhost:11434",
        output_path: str = "dataset/tailored_resumes",  # Base name without extension
        timeout: int = 300,  # Increased for thinking models
        max_retries: int = 3,
        enable_thinking: bool = True  # Enable thinking mode for supported models
    ):
        """
        Initialize the Ollama Resume Generator.
        
        Args:
            model_name: Name of the Ollama model to use
            base_url: Base URL for the Ollama API
            output_path: Base path for output file (timestamp will be added)
            timeout: Request timeout in seconds (higher for thinking models)
            max_retries: Maximum number of retries on failure
            enable_thinking: Whether to enable thinking mode (adds /think suffix)
        """
        self.model_name = model_name
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"
        self.timeout = timeout
        self.max_retries = max_retries
        self.enable_thinking = enable_thinking
        
        # Add timestamp to output filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.output_path = Path(f"{output_path}_{timestamp}.jsonl")
        
        # Ensure output directory exists
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Store session start time
        self.session_start = datetime.now().isoformat()
        
    def _build_prompt(self, resume_text: str, job_description: str) -> str:
        """Build the prompt by filling in the template."""
        prompt = PROMPT_TEMPLATE.replace("<<PASTE RESUME HERE>>", resume_text)
        prompt = prompt.replace("<<PASTE JD HERE>>", job_description)
        prompt = prompt.replace("<<PASTE JSON SCHEMA HERE>>", RESUME_SCHEMA)
        return prompt
    
    def _extract_thinking_and_response(self, response: str) -> Tuple[Optional[str], str]:
        """
        Extract thinking content and actual response from thinking model output.
        
        Returns:
            Tuple of (thinking_content, actual_response)
        """
        if not response:
            return None, ""
        
        thinking_content = None
        actual_response = response
        
        # Pattern to match <think>...</think> blocks
        think_pattern = r'<think>(.*?)</think>'
        think_match = re.search(think_pattern, response, re.DOTALL)
        
        if think_match:
            thinking_content = think_match.group(1).strip()
            # Remove the thinking block from response
            actual_response = re.sub(think_pattern, '', response, flags=re.DOTALL).strip()
        
        return thinking_content, actual_response
    
    def _call_ollama(self, prompt: str) -> Tuple[Optional[str], Optional[str]]:
        """
        Make a request to the Ollama API.
        
        Returns:
            Tuple of (response, thinking_content)
        """
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            # "options": {
            #     "temperature": 0.7,
            #     "num_predict": 4096  # Increased for thinking models
            # }
        }
        
        for attempt in range(self.max_retries):
            try:
                response = requests.post(
                    self.api_url,
                    json=payload,
                    timeout=self.timeout
                )
                response.raise_for_status()
                result = response.json()
                raw_response = result.get("response", "")
                
                # Extract thinking and actual response
                thinking, actual_response = self._extract_thinking_and_response(raw_response)
                
                return actual_response, thinking
                
            except requests.exceptions.Timeout:
                print(f"Timeout on attempt {attempt + 1}/{self.max_retries}")
            except requests.exceptions.RequestException as e:
                print(f"Request error on attempt {attempt + 1}/{self.max_retries}: {e}")
            
            if attempt < self.max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
        
        return None, None
    
    def _extract_json(self, response: str) -> Optional[Dict[str, Any]]:
        """Extract JSON from the model response."""
        if not response:
            return None
        
        # Try to find JSON in the response
        response = response.strip()
        
        # Try direct parsing first
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            pass
        
        # Try to extract JSON from markdown code blocks
        if "```json" in response:
            start = response.find("```json") + 7
            end = response.find("```", start)
            if end > start:
                try:
                    return json.loads(response[start:end].strip())
                except json.JSONDecodeError:
                    pass
        
        # Try generic code blocks
        if "```" in response:
            start = response.find("```") + 3
            # Skip language identifier if present
            newline_pos = response.find("\n", start)
            if newline_pos > start:
                start = newline_pos + 1
            end = response.find("```", start)
            if end > start:
                try:
                    return json.loads(response[start:end].strip())
                except json.JSONDecodeError:
                    pass
        
        # Try to extract JSON between curly braces
        start = response.find("{")
        end = response.rfind("}") + 1
        if start >= 0 and end > start:
            try:
                return json.loads(response[start:end])
            except json.JSONDecodeError:
                pass
        
        return None
    
    def generate_single(self, resume_text: str, job_description: str, filename: str) -> Dict[str, Any]:
        """Generate a tailored resume for a single resume-job pair."""
        start_time = datetime.now()
        prompt = self._build_prompt(resume_text, job_description)
        response, thinking_content = self._call_ollama(prompt)
        end_time = datetime.now()
        
        result = {
            "filename": filename,
            "original_resume": resume_text[:500] + "..." if len(resume_text) > 500 else resume_text,
            "job_description": job_description[:500] + "..." if len(job_description) > 500 else job_description,
            "status": "success",
            "tailored_resume": None,
            "raw_response": None,
            "thinking_content": thinking_content,  # Store the model's reasoning
            "timestamp": end_time.isoformat(),
            "processing_time_seconds": (end_time - start_time).total_seconds()
        }
        
        if response:
            parsed_json = self._extract_json(response)
            if parsed_json:
                result["tailored_resume"] = parsed_json
            else:
                result["status"] = "json_parse_error"
                result["raw_response"] = response[:2000] if len(response) > 2000 else response
        else:
            result["status"] = "api_error"
        
        return result
    
    def process_dataframe(
        self,
        df: pd.DataFrame,
        resume_col: str = "resume_text",
        job_col: str = "job_description",
        filename_col: str = "filename",
        start_idx: int = 0,
        end_idx: Optional[int] = None,
        save_every: int = 10
    ) -> None:
        """
        Process a DataFrame and generate tailored resumes.
        
        Args:
            df: DataFrame with resume and job description columns
            resume_col: Name of the resume text column
            job_col: Name of the job description column
            filename_col: Name of the filename column
            start_idx: Starting index for processing
            end_idx: Ending index for processing (None = process all)
            save_every: Save progress after every N records
        """
        if end_idx is None:
            end_idx = len(df)
        
        df_subset = df.iloc[start_idx:end_idx]
        
        successful = 0
        failed = 0
        
        batch_start_time = datetime.now()
        print(f"{'='*50}")
        print(f"Starting processing at: {batch_start_time.isoformat()}")
        print(f"Model: {self.model_name}")
        print(f"Thinking mode: {'Enabled' if self.enable_thinking else 'Disabled'}")
        print(f"Output file: {self.output_path}")
        print(f"Processing records {start_idx} to {end_idx} ({len(df_subset)} total)")
        print(f"{'='*50}\n")
        
        # Open file in append mode
        with open(self.output_path, 'a', encoding='utf-8') as fh:
            for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Generating resumes"):
                resume_text = row[resume_col]
                job_description = row[job_col]
                filename = row[filename_col]
                
                result = self.generate_single(resume_text, job_description, filename)
                result["original_index"] = idx
                result["session_start"] = self.session_start
                result["model_used"] = self.model_name
                
                # Write to file immediately
                fh.write(json.dumps(result, ensure_ascii=False) + "\n")
                
                if result["status"] == "success":
                    successful += 1
                else:
                    failed += 1
                
                # Flush periodically
                if (successful + failed) % save_every == 0:
                    fh.flush()
        
        batch_end_time = datetime.now()
        total_time = (batch_end_time - batch_start_time).total_seconds()
        
        print(f"\n{'='*50}")
        print(f"Processing complete!")
        print(f"Model:      {self.model_name}")
        print(f"Started:    {batch_start_time.isoformat()}")
        print(f"Finished:   {batch_end_time.isoformat()}")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Successful: {successful}")
        print(f"Failed:     {failed}")
        if successful + failed > 0:
            print(f"Avg time/record: {total_time/(successful+failed):.2f} seconds")
        print(f"Results saved to: {self.output_path}")
        print(f"{'='*50}")
    
    def check_ollama_status(self) -> bool:
        """Check if Ollama is running and the model is available."""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            response.raise_for_status()
            models = response.json().get("models", [])
            model_names = [m.get("name", "") for m in models]
            model_base_names = [m.split(":")[0] for m in model_names]
            
            print(f"Ollama is running. Available models: {model_names}")
            
            model_base = self.model_name.split(":")[0]
            if self.model_name in model_names or model_base in model_base_names:
                print(f"✓ Model '{self.model_name}' is available")
                if "qwen3" in self.model_name.lower():
                    print(f"  Note: Qwen3 thinking model detected - extended timeout set to {self.timeout}s")
                return True
            else:
                print(f"✗ Model '{self.model_name}' not found. Please run: ollama pull {self.model_name}")
                return False
        except requests.exceptions.RequestException as e:
            print(f"✗ Cannot connect to Ollama at {self.base_url}")
            print(f"  Error: {e}")
            print(f"  Make sure Ollama is running: ollama serve")
            return False


print("OllamaResumeGenerator class defined successfully!")

OllamaResumeGenerator class defined successfully!


### 5.3 Test Ollama Connection and Generate

In [ ]:
# Initialize the Ollama generator
generator = OllamaResumeGenerator(
    model_name="qwen3:8b",
    output_path="files/dataset/tailored_resumes/tailored_resumes",
    timeout=180,
    max_retries=3
)

# Check Ollama status
print("🔍 Checking Ollama connection...")
generator.check_ollama_status()

Ollama is running. Available models: ['qwen3-vl:8b', 'qwen3:8b', 'phi4-reasoning:plus', 'gpt-oss:20b', 'deepseek-r1:14b', 'mixtral:8x7b', 'mistral-nemo:latest', 'mistral:7b']
✓ Model 'qwen3:8b' is available
  Note: Qwen3 thinking model detected - extended timeout set to 180s


True

In [ ]:
# Process a subset of the dataset (adjust end_idx for full processing)
generator.process_dataframe(
    df,
    resume_col="resume_text",
    job_col="job_description",
    filename_col="filename",
    start_idx=0,
    end_idx=1,  # Set to None to process all records
    save_every=1
)

Starting processing at: 2025-12-01T02:23:04.655779
Model: qwen3:8b
Thinking mode: Enabled
Output file: dataset\tailored_resumes_20251201_022302.jsonl
Processing records 0 to 1 (1 total)



Generating resumes: 100%|██████████| 1/1 [00:26<00:00, 26.10s/it]


Processing complete!
Model:      qwen3:8b
Started:    2025-12-01T02:23:04.655779
Finished:   2025-12-01T02:23:30.761733
Total time: 26.11 seconds (0.44 minutes)
Successful: 1
Failed:     0
Avg time/record: 26.11 seconds
Results saved to: dataset\tailored_resumes_20251201_022302.jsonl


In [ ]:
# Read and display generated results
results_df = pd.read_json('files/dataset/tailored_resumes/tailored_resumes_20251201_022302.jsonl', lines=True)

print(f"📊 Generated {len(results_df)} tailored resumes")
print(f"\n📈 Status Distribution:")
print(results_df['status'].value_counts())

# Show sample of successful result
successful = results_df[results_df['status'] == 'success']
if len(successful) > 0:
    sample = successful.iloc[0]
    print(f"\n" + "=" * 50)
    print(f"SAMPLE TAILORED RESUME")
    print("=" * 50)
    print(f"Filename: {sample['filename']}")
    print(f"\nJSON Output:")
    print(json.dumps(sample['tailored_resume'], indent=2)[:1000] + "...")

Generated 1 tailored resumes

Status distribution:
status
success    1
Name: count, dtype: int64

=== Sample Tailored Resume ===
Filename: 00_Willie_Ellis_Go_Python.docx

Tailored Resume JSON:
{
  "personal_information": {
    "name": "Willie Ellis",
    "email": "willieellis0177@gmail.com",
    "phone": "760 995 2578",
    "location": "Buffalo, New York",
    "socials": [
      {
        "name": "LinkedIn",
        "link": "http://www.linkedin.com/in/willieellisa6b492212"
      }
    ]
  },
  "summary": "Detail-oriented and dependable individual with a strong work ethic, capable of maintaining a safe and organized work environment. Eager to contribute to production goals and ensure quality standards.",
  "experiences": [
    {
      "designation": "Textiles Assistant",
      "companyName": "Goodwill Industries of Northwest NC",
      "location": "Brevard, NC",
      "start_date": "2023-08-01",
      "end_date": "Present",
      "points": [
        "Sorting clothing with attention to q

In [ ]:
# Display results dataframe info
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   filename                 1 non-null      object        
 1   original_resume          1 non-null      object        
 2   job_description          1 non-null      object        
 3   status                   1 non-null      object        
 4   tailored_resume          1 non-null      object        
 5   raw_response             0 non-null      float64       
 6   thinking_content         0 non-null      float64       
 7   timestamp                1 non-null      datetime64[ns]
 8   processing_time_seconds  1 non-null      float64       
 9   original_index           1 non-null      int64         
 10  session_start            1 non-null      object        
 11  model_used               1 non-null      object        
dtypes: datetime64[ns](1), float64(3), int64(

---

## 6. ☁️ Gemini Batch API Processing

This section uses **Google's Gemini API** for large-scale batch processing of resume generation requests.

### Advantages of Batch Processing:
- **Cost Efficient:** 50% discount on batch API requests
- **Scalable:** Process thousands of requests in parallel
- **Reliable:** Built-in retry and error handling

### 6.1 Define Prompts for Batch API

In [ ]:
# JSON Schema for tailored resume output
RESUME_SCHEMA = '''{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"type":"string"},"location":{"type":"string"},"socials":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]}},"required":["name","email","phone","location"]},"summary":{"type":"string"},"experiences":{"type":"array","items":[{"type":"object","properties":{"designation":{"type":"string"},"companyName":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"points":{"type":"array","items":[{"type":"string"}]}},"required":["designation","companyName","location","start_date"]}]},"education":{"type":"array","items":[{"type":"object","properties":{"institution":{"type":"string"},"degree":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"gpa":{"type":"string"}},"required":["institution","degree","location","start_date","gpa"]}]},"skills":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"data":{"type":"array","items":[{"type":"string"}]}},"required":["name","data"]}]},"projects":{"type":"array","items":[{"type":"object","properties":{"projectName":{"type":"string"},"caption":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"url":{"type":"string"},"projectDetails":{"type":"array","items":[{"type":"string"}]},"externalSources":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]},"technologiesUsed":{"type":"array","items":[{"type":"string"}]}},"required":["projectName","location","projectDetails"]}]},"certifications":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"issuing_organization":{"type":"string"},"issue_date":{"type":"string"},"expiration_date":{"type":"string"},"credential_id":{"type":"string"},"url":{"type":"string"}},"required":["name","issuing_organization","issue_date","expiration_date","credential_id","url"]}]},"awards":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"extracurricular_achievements":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"languages":{"type":"array","items":[{"type":"object","properties":{"language":{"type":"string"},"proficiency":{"type":"string"}},"required":["language","proficiency"]}]}},"required":["personal_information","education","skills","extracurricular_achievements"]}'''


SYSTEM_PROMPT = """You create a tailored resume based on the job description.

Your task:
1. Read the RESUME_TEXT.
2. Read the JOB_DESCRIPTION.
3. Use only the information inside these two.
4. Follow the SCHEMA exactly.
5. Write a tailored resume in JSON using the SCHEMA.
6. Do not output anything outside the JSON.
7. If a field is missing in the resume, write a short, safe placeholder that fits the job.

"""

PROMPT_TEMPLATE = """
RESUME_TEXT:
\"\"\"
<<PASTE RESUME HERE>>
\"\"\"

JOB_DESCRIPTION:
\"\"\"
<<PASTE JD HERE>>
\"\"\"

SCHEMA:
\"\"\"
<<PASTE JSON SCHEMA HERE>>
\"\"\"

Create a tailored resume in JSON following the SCHEMA exactly.
Use only content from the RESUME_TEXT but rewrite it to match the JOB_DESCRIPTION.
Do not add extra lines or explanation.
Output only JSON.
"""

In [ ]:
def _build_USER_prompt(resume_text: str, job_description: str) -> str:
    """Build the user prompt by filling in the template."""
    prompt = PROMPT_TEMPLATE.replace("<<PASTE RESUME HERE>>", resume_text)
    prompt = prompt.replace("<<PASTE JD HERE>>", job_description)
    prompt = prompt.replace("<<PASTE JSON SCHEMA HERE>>", RESUME_SCHEMA)
    return prompt

### 6.2 Generate Batch Request Files

Create JSONL files with batch requests for the Gemini API.

In [ ]:
# Load data and prepare batch requests
DATA_DIR = ".files/dataset/resume-data-with-job-description/"
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')

# Column names
resume_col = "resume_text"
job_col = "job_description"

def generate_request_file(batch_name: str, df_subset: pd.DataFrame) -> None:
    """Generate a JSONL file with batch requests for the Gemini API."""
    output_filename = f"batch_requests_{batch_name}.jsonl"
    output_path = Path(f"dataset/batch_requests/{output_filename}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    requests = []
    print(f"\n🔄 Generating requests for: {output_filename}")
    
    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Processing {batch_name}"):
        user_content = _build_USER_prompt(row[resume_col], row[job_col])
        
        request_entry = {
            "key": f"request-{batch_name}-idx{idx}",
            "request": {
                "contents": [{"role": "user", "parts": [{"text": user_content}]}],
                "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
                "generationConfig": {"responseMimeType": "application/json", "temperature": 0.2}
            }
        }
        requests.append(request_entry)
    
    # Write to JSONL
    with output_path.open('w', encoding='utf-8') as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
    
    print(f"✅ Created {output_filename} with {len(requests)} requests")

# Generate batch file (adjust indices as needed)
batch_key = "batch-4"
df_subset = df.iloc[521:1565]
generate_request_file(batch_key, df_subset)


Generating requests for  -> batch_requests_batch-4.jsonl


Processing batch-4: 100%|██████████| 1044/1044 [00:00<00:00, 11106.29it/s]

Success: Created batch_requests_batch-4.jsonl with 1044 requests.


In [ ]:
# Verify dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1565 entries, 0 to 2126
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1565 non-null   object
 1   filetype         1565 non-null   object
 2   resume_text      1565 non-null   object
 3   has_images       1565 non-null   bool  
 4   job_description  1565 non-null   object
dtypes: bool(1), object(4)
memory usage: 62.7+ KB


### 6.3 Configure Gemini API Client

Set up authentication for the Google GenAI client.

#### Option 1: Configure with Gemini API key

In [ ]:
# Option 1: Configure with Gemini API key
client = genai.Client(api_key="YOUR_API_KEY_HERE")
print("✅ GenAI configured with API key!")

# Note: Replace with your actual API key or use environment variable
print("⚠️ API key configuration required - update this cell with your key")

GenAI configured successfully with gemini api key!


#### Option 2: Configure with Vertex AI (Google Cloud)

In [ ]:
# Option 2: Configure with Vertex AI (Google Cloud)
# Set the path to your service account key file
CREDENTIALS_PATH = "gen-lang-client-0465995346-fd97340f8167.json"

if os.path.exists(CREDENTIALS_PATH):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(CREDENTIALS_PATH)
    print(f"✅ Service account credentials loaded from: {CREDENTIALS_PATH}")
else:
    print(f"⚠️ Credentials file not found: {CREDENTIALS_PATH}")

In [ ]:
# Initialize Vertex AI client
client = genai.Client(
    vertexai=True,
    project='gen-lang-client-0465995346',
    location='us-central1'
)
print("✅ GenAI configured with Vertex AI!")

GenAI configured successfully with vertex api key!


### 6.4 Submit and Monitor Batch Jobs

In [ ]:
# Define batch file path
batch_file = 'files/dataset/batch_requests/batch_requests_batch-4[521-1535].jsonl'
print(f"📄 Batch file: {batch_file}")

In [ ]:
# Upload batch file to Google Cloud
uploaded_file = client.files.upload(
    file=batch_file,
    config=types.UploadFileConfig(display_name='my-batch-requests', mime_type='jsonl')
)
print(f"✅ Uploaded file: {uploaded_file.name}")

# Reference: Previous uploads
# batch-1: files/igujaoz0own5 → batches/mpfcsvvz5f3demju35es8w4tlqzunsjncxhn
# batch-4: files/saw7wfyrzg3p

In [ ]:
# Configure batch job parameters  for vertex AI
display_name = 'batch-upload-job-batch_requests_batch-4'
file_name = 'gs://harsha-dump/batch_requests_batch-4.jsonl'

print(f"📋 Job name: {display_name}")
print(f"📁 Source file: {file_name}")

In [ ]:
# Create and submit batch job
file_batch_job = client.batches.create(
    model="gemini-2.5-flash",
    src=file_name,
    config={'display_name': display_name}
)
print(f"✅ Created batch job: {file_batch_job.name}")

Created batch job: projects/982088254342/locations/us-central1/batchPredictionJobs/8949234702531690496


In [ ]:
# Check batch job status
job_name = "projects/982088254342/locations/us-central1/batchPredictionJobs/2288129378674016256"

completed_states = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED', 'JOB_STATE_EXPIRED'}

print(f"🔍 Checking status for: {job_name}")
batch_job = client.batches.get(name=job_name)
print(f"📊 Current state: {batch_job.state.name}")

if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"❌ Error: {batch_job.error}")

Polling status for job: projects/982088254342/locations/us-central1/batchPredictionJobs/2288129378674016256
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING


In [ ]:
# Cancel batch job if needed
# client.batches.cancel(name=job_name)
# print("⚠️ Batch job cancelled")

In [ ]:
# Download batch results when job completes
job_name = "batches/mpfcsvvz5f3demju35es8w4tlqzunsjncxhn"
batch_job = client.batches.get(name=job_name)

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    if batch_job.dest and batch_job.dest.file_name:
        result_file_name = batch_job.dest.file_name
        print(f"📥 Downloading results from: {result_file_name}")
        
        file_content = client.files.download(file=result_file_name)
        
        # Save to local file
        output_path = Path(f"files/dataset/batch_results/{result_file_name.replace('/', '_')}.jsonl")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with output_path.open('wb') as f:
            f.write(file_content)
        
        print(f"✅ Results saved to: {output_path}")
    else:
        print("⚠️ No file results found")
else:
    print(f"⚠️ Job state: {batch_job.state.name}")

### 6.5 Process Batch Results

Parse and clean the batch API responses, then merge with the original dataset.

In [ ]:
# Add ID column for tracking
df['id'] = df.index
print(f"✅ Added ID column to dataframe")

In [ ]:
# Save and reload dataframe with ID
DATA_DIR = ".\files\dataset\resume-data-with-job-description"
df.to_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"✅ Dataframe saved and reloaded: {len(df)} records")

In [ ]:
# Load batch results from multiple files
batch_result_files = [
    './files/dataset/batch_results/files_batch-mpfcsvvz5f3demju35es8w4tlqzunsjncxhn.jsonl',
    './files/dataset/batch_results/batch-output_prediction-model-2025-12-02T21_47_49.533170Z_predictions.jsonl'
]

list_of_dfs = []
for path in batch_result_files:
    if os.path.exists(path):
        temp = pd.read_json(path, lines=True)
        list_of_dfs.append(temp)
        print(f"📄 Loaded {len(temp)} records from: {os.path.basename(path)}")

# Combine all results
results_df = pd.concat(list_of_dfs, ignore_index=True)
results_df = results_df.drop(columns=['request', 'status', 'processed_time'], errors='ignore')

print(f"\n📊 Total combined records: {len(results_df)}")
results_df.info()

Loaded 521 records from first batch result
Loaded 1044 records from second batch result
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   response  1565 non-null   object
 1   key       1565 non-null   object
dtypes: object(2)
memory usage: 24.6+ KB


,response,key
0,"{'candidates': [{'index': 0, 'finishReason': '...",request-batch-1-idx0-1
1,"{'responseId': 'FEUvaf3SLsnYqtsPka-dGA', 'usag...",request-batch-1-idx1-2
2,"{'candidates': [{'index': 0, 'finishReason': '...",request-batch-1-idx2-3
3,"{'responseId': 'FUUvabHPMOD6qtsPicKv-QY', 'mod...",request-batch-1-idx3-4
4,"{'candidates': [{'finishReason': 'STOP', 'cont...",request-batch-1-idx6-5


In [ ]:
def extract_indices(key: str) -> Optional[int]:
    """Extract the original index from the batch request key."""
    pattern = re.compile(r'idx(\d+)')
    match = pattern.search(key)
    return int(match.group(1)) if match else None

In [ ]:
def _extract_json(response: str) -> Optional[Dict[str, Any]]:
    """Extract JSON from model response with multiple fallback strategies."""
    if not response:
        return None
    
    response = response.strip()
    
    # Strategy 1: Direct parsing
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass
    
    # Strategy 2: Extract from ```json code blocks
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end > start:
            try:
                return json.loads(response[start:end].strip())
            except json.JSONDecodeError:
                pass
    
    # Strategy 3: Extract from generic code blocks
    if "```" in response:
        start = response.find("```") + 3
        newline_pos = response.find("\n", start)
        if newline_pos > start:
            start = newline_pos + 1
        end = response.find("```", start)
        if end > start:
            try:
                return json.loads(response[start:end].strip())
            except json.JSONDecodeError:
                pass
    
    # Strategy 4: Extract between curly braces
    start = response.find("{")
    end = response.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(response[start:end])
        except json.JSONDecodeError:
            pass
    
    return None

In [ ]:
# Process batch results and extract tailored resumes
processed_results = []
errors = []

for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Processing results"):
    key = row['key']
    extracted_index = extract_indices(key)
    
    result = {
        'key': key,
        'original_index': extracted_index,
        'status': 'success',
        'tailored_resume': None,
        'raw_response': None,
        'error': None,
        'found_in_df': extracted_index in df.index.tolist() if extracted_index else False
    }
    
    try:
        response = row.get('response', {})
        candidates = response.get('candidates', [])
        
        if not candidates:
            result['status'] = 'no_candidates'
            result['error'] = 'No candidates in response'
        else:
            parts = candidates[0].get('content', {}).get('parts', [])
            if not parts:
                result['status'] = 'no_parts'
                result['error'] = 'No parts in content'
            else:
                text = parts[0].get('text', '')
                result['raw_response'] = text
                
                cleaned_json = _extract_json(text)
                if cleaned_json:
                    result['tailored_resume'] = cleaned_json
                else:
                    result['status'] = 'json_parse_error'
                    result['error'] = 'Failed to parse JSON'
    except Exception as e:
        result['status'] = 'error'
        result['error'] = str(e)
        errors.append({'key': key, 'error': str(e)})
    
    processed_results.append(result)

processed_df = pd.DataFrame(processed_results)

# Display summary
print(f"\n" + "=" * 50)
print("PROCESSING SUMMARY")
print("=" * 50)
print(f"Total records: {len(processed_df)}")
print(f"\n📊 Status Distribution:")
print(processed_df['status'].value_counts())
print(f"\n✅ Found in original df: {processed_df['found_in_df'].sum()}")

Mapping original indices: 100%|██████████| 1565/1565 [00:00<00:00, 8792.20it/s]


=== Processing Summary ===
Total records: 1565
Status distribution:
status
success             1530
json_parse_error      22
no_parts              13
Name: count, dtype: int64

Found in original df: 1565
Not found in original df: 0


In [ ]:
# Count successful extractions
successful = processed_df[processed_df['status'] == 'success']
print(f"✅ Successful JSON extractions: {len(successful)}")

Successful extractions: 1530

=== Sample Tailored Resume ===
Key: request-batch-1-idx0-1
Original Index: 0

Tailored Resume JSON (first 500 chars):
{
  "personal_information": {
    "name": "Willie Ellis",
    "email": "willieellis0177gmailcom",
    "phone": "760 995 2578",
    "location": "Buffalo New York",
    "socials": [
      {
        "name": "LinkedIn",
        "link": "httpswwwlinkedincominwillieellisa6b492212"
      }
    ]
  },
  "summary": "Highly motivated professional with 7 years of work experience, demonstrating strong commitment, dependability, and hard work. Proven ability to implement process improvements to optimize effi...


In [ ]:
# Merge tailored resumes back to original dataframe
df['tailored_resume'] = None
count = 0
not_found = 0

for idx, row in tqdm(successful.iterrows(), total=len(successful), desc="Merging results"):
    original_idx = row['original_index']
    
    if original_idx in df['id'].values:
        df.loc[df['id'] == original_idx, 'tailored_resume'] = [row['tailored_resume']]
        count += 1
    else:
        not_found += 1

print(f"\n" + "=" * 50)
print("MERGE SUMMARY")
print("=" * 50)
print(f"✅ Successfully merged: {count}")
print(f"⚠️ Not found in df: {not_found}")
print(f"📊 Total with tailored_resume: {df['tailored_resume'].notna().sum()}")

# Remove rows without tailored resumes
print(f"\n📊 Before filtering: {len(df)} rows")
df = df[df['tailored_resume'].notna()]
df = df.drop(columns=['has_images'], errors='ignore')
print(f"📊 After filtering: {len(df)} rows")

Mapping tailored resumes to df: 100%|██████████| 1530/1530 [00:00<00:00, 2359.29it/s]


=== Mapping Summary ===
Successfully mapped: 1530
Not found in df: 0
Total rows with tailored_resume: 1530
Before removing nulls: 1565 rows
After removing nulls: 1530 rows


### 6.6 Save Final Dataset

Export the combined dataset with tailored resumes.

In [ ]:
# Save final dataset in multiple formats
DATA_DIR = "./files/dataset/resume-data-with-job-description"

df.to_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df.to_json(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl'), orient='records', lines=True, force_ascii=False)
df.to_csv(os.path.join(DATA_DIR, 'final_resume_dataset.csv'), index=False, encoding='utf-8-sig')

print("✅ Final dataset saved:")
print(f"   📄 {DATA_DIR}/final_resume_dataset.parquet")
print(f"   📄 {DATA_DIR}/final_resume_dataset.jsonl")
print(f"   📄 {DATA_DIR}/final_resume_dataset.csv")

# Reload for verification
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"\n📊 Verified: {len(df)} records loaded")

---

## 7. 📝 Training Dataset Preparation

Convert the processed data into the format required for fine-tuning (chat-style JSONL).

### Dataset Format:
```json
{
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "{...JSON resume...}"}
  ]
}
```

In [ ]:
# Load the final dataset
DATA_DIR = "./files/dataset/resume-data-with-job-description"
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"📊 Loaded {len(df)} records for training data preparation")

In [ ]:
# Verify dataframe structure
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1530 entries, 0 to 2126
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1530 non-null   object
 1   filetype         1530 non-null   object
 2   resume_text      1530 non-null   object
 3   job_description  1530 non-null   object
 4   id               1530 non-null   int64 
 5   tailored_resume  1530 non-null   object
dtypes: int64(1), object(5)
memory usage: 83.7+ KB


### 7.1 Convert to Chat Format

In [ ]:
# Convert dataset to chat format for fine-tuning
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Converting to chat format"):
    user_content = _build_USER_prompt(row["resume_text"], row["job_description"])
    resume = row["tailored_resume"]
    
    # Handle numpy arrays and other non-string types
    if isinstance(resume, np.ndarray):
        resume = resume.tolist()
        if len(resume) == 1:
            resume = resume[0]
    if not isinstance(resume, str):
        resume = str(resume)
    
    result = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": resume}
        ]
    }
    results.append(result)

# Save training dataset
output_filename = "final_training_dataset"
output_path = Path(f"dataset/training/{output_filename}.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open('w', encoding='utf-8') as f:
    for req in results:
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

print(f"✅ Created {output_filename}.jsonl with {len(results)} training examples")

Extracting tailored resumes: 100%|██████████| 1530/1530 [00:00<00:00, 2825.50it/s]



Success: Created final_training_dataset with 1530 requests.


### 7.2 Clean Dataset (Fix JSON Formatting)

In [ ]:
import json
import re

def clean_python_to_json(text):
    # Replace Python None/True/False
    text = text.replace("None", "null")
    text = text.replace("True", "true")
    text = text.replace("False", "false")

    # Convert single quotes to double quotes carefully
    text = re.sub(r"'", '"', text)

    # Remove numpy arrays
    text = re.sub(r'array\((.*?)\)', r'\1', text)

    # Remove dtype info
    text = re.sub(r'dtype=object', '', text)

    # Remove trailing commas before }
    text = re.sub(r',\s*}', '}', text)

    # Remove trailing commas before ]
    text = re.sub(r',\s*\]', ']', text)

    return text


cleaned_dataset = []

with open("files/dataset/training/final_training_dataset.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)

        # Clean assistant message
        assistant_msg = item["messages"][-1]["content"]
        cleaned = clean_python_to_json(assistant_msg)

        # Replace assistant content
        item["messages"][-1]["content"] = cleaned

        cleaned_dataset.append(item)

# Write new clean dataset
with open("files/dataset/training/final_training_dataset_cleaned.jsonl", "w") as f:
    for item in cleaned_dataset:
        f.write(json.dumps(item) + "\n")

print("Dataset cleaned! Output saved to final_training_dataset_cleaned.jsonl")

Dataset cleaned! Output saved to train_clean.jsonl


---

## 8. 🧪 Base Model Testing

Test the base Qwen3 model before fine-tuning to establish a baseline.

### Model Configuration:
- **Model:** Qwen3-4B-Instruct-2507
- **Quantization:** 4-bit (NF4) with double quantization
- **Attention:** Flash Attention 2 (if available)

In [ ]:
# Check device availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Using device: {device}")

if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

'cuda'

### 8.1 Load Base Model with Quantization

In [ ]:
# Load base model with 4-bit quantization
model_name = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Check Flash Attention availability
flash_attn_available = importlib.util.find_spec("flash_attn") is not None
use_flash_attn = (flash_attn_available and torch.cuda.is_available() and 
                  torch.cuda.get_device_properties(0).major >= 8)

print(f"🔄 Loading model: {model_name}")
print(f"⚡ Flash Attention: {'Enabled' if use_flash_attn else 'Disabled'}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2" if use_flash_attn else "eager",
    torch_dtype=torch.bfloat16,
)

print("✅ Model loaded successfully!")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.02s/it]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated Resume:

Content saved to 'generated_resume.txt'


### 8.2 Test Base Model Generation

Generate a sample resume with the base model (before fine-tuning).

In [ ]:
# prepare the model input
messages = [
    {"role": "system", "content": "You create a tailored resume based on the job description.\n\nYour task:\n1. Read the RESUME_TEXT.\n2. Read the JOB_DESCRIPTION.\n3. Use only the information inside these two.\n4. Follow the SCHEMA exactly.\n5. Write a tailored resume in JSON using the SCHEMA.\n6. Do not output anything outside the JSON.\n7. If a field is missing in the resume, write a short, safe placeholder that fits the job.\n\n"},
    {"role": "user", "content": "\nRESUME_TEXT:\n\"\"\"\nPHANI SETTY 214 9235723 settyphanigmailcom Summary A handson programmer in userinterface design and development with over 20 years of experience and a proven track record in shipping web experiences for singlepage applications hybrid apps online stores and interacting market content Extensive experience in Architecting and developing for large and distributed frontend codebases as well as restructuring legacy systems to reduce bloat increase modularity and speed up future development iterations I have worked with a variety of web application stacks along with their respective view and templating systems When I am not coding I am involved in UXside of things like sketching out wireframes and designs on paper or whiteboard creating highfidelity designs and clickable prototypes and usertesting them I have developed innovative products and express brands through strategically driven design and interactive models 15 years of handson experience in leading teams building highperforming teams of onsite remote and offshore developers and designers Afterhours Im an avid learner of new technologies through personal projects as well as writing a book I was a Certified PMP20062009 and a Certified ScrumMasterpresent I bring people and technology together through effective communication and leadership Im equally comfortable discussing business requirements with product managers architects and APIs with developers Ive designed developed and delivered technology solutions in both traditional Waterfall and Agile development environments as well as transitioning between them I Have worked extensively in Healthcare Education Management Consulting domains I have strong technical and communication skills Core Competencies CSS Architecture I Frontend Architecture I Design Systems I Adaptive responsive design I wireframing Prototyping browser device debugging I Pageload Performance tuning I Mentoring and Training I Agile software development Technical Competencies Clientside UI JavaScript JQuery ReactJS redux Angular Websockets RequireJS HeatmapJS TypeScript HTMS CSS pre post processing SVG Backbone Handlebars D3 Environments ASPNET Core Web Forms and MVC Nodejs Java Grails PHP Git Github Gitlab CVS TFS Tools Visual Studio Code IntelliJ Adobe CC suite Education BE Computer Science Amravati University 1996 BSIT from Grantham University2017 MSSoftware Engineering from Walden University2021 Certifications Certified scrum master2018 Certified PMP2006 Six sigmagreen belt 2002 Projects CBRE UI Lead July 2018 Till Date Dallas tx Responsibilities As the onshore lead for a large consulting team I oversaw coding standards related to Angular 28 CSS JavaScript I wrote prototypes as well as augmented teams to facilitate project completion My UI organization included creating and administering training programs and overseeing code reviews for a global team of developers at various skill levels along with working closely with clients being empathetic to their needs all the while interpreting them in a cognitive experience that makes sense to their customer Accomplished Web project objectives by establishing clear understanding of project requirements Involved in designing and developing the components using HTML CSS JavaScript Bootstrap SASS Angular8 Flex and NodeJS Involved in implementing various screens for the frontend using Angular and used various public libraries from NPM Node Package Manager Collaborate crossfunctionally to develop research plan user flows information architecture and wireframes and work with Agile Scrum teams Managed a growing team of UI Engineers focused on building great experiences teaching and implementing excellence Visualize large data and develop dashboards As a lead Have good ability to analyze problems find solutions and implement them to tight deadlines on time I have good experience writing technical briefs technical specifications and generating costingtimings for projects Dealer socket Tech Leadui apr2018 July 2018 irving tx Responsibilities As a Lead I am responsible for making sure a quality and a welldocumented and test driven code is developed Extensive experience in building many software solutions with particular emphasis on clientside code Web Apps and Native Mobile Apps Specialized in architecting UI frameworks and creating custom reusable user interface components Create responsive landing pages and email templates for product communications Create and manage paid social media ad campaigns to drive lead generation Develop maintain systems utilizing HTML CSS JavaScriptjQuery Develop maintain systems utilizing client side frameworks and libraries Work closely with product owners backend C developers and UX designers to implement new features Mobile Applications Web Single Page Applications UI Components development WebComponents React Redux ReduxSaga ES Flow Jest Enzyme Babel NodeJS Webpack CI with Bitbuckets Pipeline and Docker containers HarMAN internationalTech leadui Dec2017 April 2018Plano tx Responsibilities As a Lead I am responsible for making sure a quality and a welldocumented code is developed Prototypingarchitecting and implementation of UI Secure clustering components and pages using ReactRedux Bootstrap UI RESTfetch Knex Bookshelf SASS GitGithub Webpack CreatingRefactoring ReactRedux reusable components for integration into encrypting dependencies Design and develop reusable Angular 4 Node Package Modules Provide guidance on technical architecture using HTML5JavaScript web technologies Builds responsive web user interfaces that ensure seamless user experience across desktop and mobile platforms Builds fullduplex embedded web applications that control hardware devices in realtime using web sockets Establishes UI development process for embedded devices Unit testing with all elements of ReactRedux project by using Jest Enzyme Adjustment desktop web apps for mobilesize devices such as smartphones and tablets Mock servers coding on Python and Golang for unit testing on local environment Integration testing with Selenium IBMUI ArchitectJuly 2017 dec2017 Santa Clara CA Responsibilities DesignArchitect support and lead both offshore and onsite teamsabout 10 React and AngularUI developers as part of refactoring an IBMs flagship product Worked with modules like MongoDB and mongoose for database persistence using Nodejs to interact with mongodb Worked with unit testing of javascript applications using Karma Jasmine apimocker Jest enzyme snion Tasks include making sure a quality and a welldocumented code is developed and ESLint errors are fixed unit tests are written and the CI builds are through for code merge Interpret clients needs and ability to architect design and develop solutions with high visual impact to get the clients message across I am responsible for user interface strategy planning development and delivery across the practice My duties are to maintain consistent design guidelines best practices and standards and project methodologies My role as the Architect for UI is to work with product owners and stakeholders to present and create solutions to visualize design and deliver style guides prototypes and assets for the client with full compliance of client rules and guidelines Optum technologiesuhg dev LEAD April2016 May 2017 Minneapolis Plano Responsibilities Evaluate and implement SEO friendly infrastructures in HTML and Javascript to new and existing applications Migrating legacy Angular 14 components to higher versions at the moment Front end architecture and development of largescale Angular application in the AEM environment Developing a scaffolding system to build the frontend using Gulp along with integrating SASS preprocessing and Bootstrap library SPAs using AngularJS 1x including factory services and directives consumption of web services Consulting on UX and visual design options Recommending design and technical solutions based on user requirements and business needs Develop poling and cross team communication ApplicationPCTC Developed documentation testing standards Implementation of Bootstrap Foundation and Jquery frameworks HealthPartners Senior Web DeveloperUI Lead Jan 2016 March 2016 Minneapolis HealthPartners is an integrated nonprofit health care provider and health insurance company located in Bloomington Minnesota offering care coverage research and education to its members patients and the community Duties include working collaboratively with the team and management on designing and delivering complex crossbrowser applications and maintaining existing products Responsibilities I was programming Widgets and Applications for wwwHealthPartnerscom using Angularjs Javascript and BootStrap Design development and testing phases of Software Development using AGILE Methodology and Test Driven Development TDD Involvement in all stages of Software development life cycle including Analysis development Implementation testing and support Involved in development of User Interface using HTML5 CSS3 JavaScript and jQuery AJAX JSON Developed single page web application using JavaScript framework Worked with CrossBrowser Compatible issues Created reusable templates and style sheets based on UI standards and guidelines Performed Functional tasks using specifications and wireframes Extensively used Debugging Cascading Style Sheets to change the styles now and in the future Designed and implemented the UI with extensive use of JavaScript JSON and Ajax Designed and developed basic user interfaces to HealthPartners web services by analyzing business requirements and priorities Provides code and web design reviews Integrated applications to web services via server scripting and database architecting Troubleshoots development and production incidents across multiple environments and operating platforms Medtronic UI Lead Nov 2014 Dec 2015 Minneapolis Responsibilities I was responsible for Architecting used combination of Event MVC and AMD patterns designing and programming the frontend for an Electro Cradio Gram waveforms rendering mobile application using Javascript and HTML 5 canvas I have developed a handy UI widget library using pure Javascript for our internal app developers to create UI elements dynamically Managed software development operational and client integration projects Created a complete test bed for the UI usingstubbing the server using Node JS I have designed the widget library in the AMD pattern using RequireJS I have also developed various reports and charts using HTML Canvas HTML SVG D3JS and SVGjs by passing JSON objects or Arrays as input both for mobile and web applications I have worked extensively on Ajax and JavaScript Websockets I have revamped an existing single thread application that had heavy computational data in the UI to a light weight application using Web workers Have been working on Jasmine and Chutzpah for my unit testing as we develop using TDD approach in Visual StudioXamarin environment Worked extensively with jQuery HTML4 and CSS Developed prototypes and mokups using Adobe Fireworks Edge and Balsamiq Developed user friendly and attractive UI Have also hand coded web templates using HTML5 Javascript Bootstrap and CSS3 Harvard Business Publishing Global engagement manager UI Lead Sep 2010 October 2014 Boston MA Responsibilities Managing the global vendors offering services in content development and translationlocalization Gathering and analyzing requirements and writing SOWs and scope documents along with leading the project from concept to completion Responsible for User Interface design and development for Harvard Business Publishings portals and products like LeadershipDirectorg WCMS and Harvard Manage Mentor using Adobe Photoshop CS5 Adobe Flash CS5 Illustrator HTML5 Javascript Angular JS JQuery CSS3 and AJAX I was responsible for designing solutions that improved user experience and supported graphic resource needs for various products for Harvard Business Publishing Boston MA I was also responsible for creatinguser personas creating wireframes visual mockups UI specs and flash designs for service window applications Created design strategy and implemented in various UIUX projects I have conducted research and data evaluation on interactive products and created graphics animations using flash and after effects for various service window applications Responsible for reviewing creative work provide art direction and design feedback when working with junior designers and developers My technical and creative Skills helped me to work on multiple projects simultaneously I have worked with product management and engineering teams to ensure that the graphics and layout designs meet customer requirements and implementation constraints GE Sr UI Developerproject managerTech lead Oct 2004 Sep 2010 Albany New York Responsibilities Gathering and analyzing requirements and writing SOWs and scope documents along with leading the project from concept to completion Responsible for User Interface Design for web applications and learning portals using Adobe Photoshop CS5 Adobe Flash CS5 Illustrator HTML5 Javascript CSS3 AJAXand the clients preferred content management tools like Knet Participate in early sprints of agile methods Work ahead of sprint and keep the work ready for the development team for development Created the rich user experience and user interface design for their Archeological data collection application for constructions process Involved in the hiring process and recruited worldclass talent user interface development group Support flash action script for interactive service window applications across the apps team Involved in the research and discover phase of the apps development for client and user research Created the rich presentations for the corporate initiatives United Nations Development Program UI Dev Jul 2002 Sep 2004 New York Responsibilities Responsible for User Interface Design using Adobe Photoshop Flash HTML CSS JavaScript and the companys custom web development and content management tools Involve in daily scrums create IA Visual design for agile process Supported interaction design visual design for web products and produced images banner logo poster brochure using softwares like Photoshop Image ready Illustrator In Design IMI Web designer Jan 1999 Jun 2002 Hyderabad India Responsibilities Responsible for User Interface Design and development for various mobilewebdesktop applications for Vodafone I have also created the interface for WAP based application called VOOP Virtual Office on Phone the application enables the users to remotely access their PCs from mobiles to send mails edit documents etc I have also produced entire UI kit for transmission tower designing applications using Photoshop and icon maker tools This was challenging back then as these tools were coded in VC and VC supported transparency only in 256 color ico formats Fountainhead Design Studios Graphic Designer Jan 1997 Jan 1999 Hyderabad India Responsibilities I have created 120 animated greeting cards for Archies online portal using Macromedia Flash 40 on iMac I was involved in the process right from conceptualizing to completion of the greeting cards I have also produced about 150 printed greeting cards using Photoshop Bryce 3D and Corel Draw I have also designed and developed more than 50 websites using basic HTML Macromedia Flash 40 and Dreamweaver 20\n\"\"\"\n\nJOB_DESCRIPTION:\n\"\"\"\nExtra Help Stagehand - 50 vacancies jobs in United StatesOverviewCompanyThis job has closed.APPLY to similar jobsUniversity of Illinois Springfield · 4 weeks agoExtra Help Stagehand - 50 vacanciesSpringfield, ILFull-timeOnsiteEntry Level$20.07/hr - $20.07/hrThe University of Illinois Springfield is seeking Extra Help Stagehands to support their theatrical productions. The role involves coordinating and executing various tasks related to the preparation, operation, and maintenance of event equipment, stage lighting, sound systems, and theatrical scenery, ensuring a safe and efficient working environment.EducationUniversitiesResponsibilitiesAttendance (paid) at employer-required safety training sessionsLoad & unload theatrical equipment into and out of vehicles as required including stacking and unstacking of equipmentMove theatrical equipment into and out of venue storage areasPerform all work required to operate and maintain the safety, trim, balance, and proper rigging of the counterbalanced fly system and associated winched cables, pin-rail equipment and pick-linesPerform such technical specialties as the splicing of cables and ropes, the use of stage weights and braces, the maintenance and correct use of tie lines and pick lines, the maintenance and repair of curtains, and the periodic inspection of curtains in storage to prevent damage.Operate mechanical systems such as pit-lifts, orchestra shell winches, chain motors, etc. Operating of chain motors does NOT also include functions performed by EXTRA HELP RIGGERS.Operate personnel-lift devicesHang, connect to the dimming system, lamp, focus and color theatrical lighting instruments for general and specific usesMove and place audio gear such as microphones, speakers, monitors, control boards and cable as requiredInstall, remove, operate and reconfigure stage curtains, portable flooring, scenery, props and other theatrical items as required.Sweeping and mopping of stage area as required for safety and proper audience presentationMaintain, move, set up and operate spotlights as requiredMaintain and operate lighting and audio control systemsInstall, maintain and operate as needed equipment hung from counter-weighted battens.Setup, maintain (including laundry), repair and place back into storage costumes, wigs and other items worn by performers including towels and associated fabric items. Also assist performers with putting on and taking off costume items.Perform other duties commonly associated with stage operations as requiredQualificationTheatrical lighting operationSound system operationStage riggingCostume maintenancePhysical staminaSafety complianceTeam collaborationAttention to detailRequiredBasic entry-level knowledge of a back-stage theatrical working environment including terminology & standard operating procedures usually acquired through formal education in the performing arts field or previous volunteer/work experience as a stage crew member in a high school, college, community theatre, music club, regional theatre, or professional performance venue.Normal ability to hear, see, speak, smell in a backstage environment which at times can be darkened, crowded, and noisy with multiple hazards.Ability to move quickly unaided.Ability to climb stairs, ladders and work at heights.Ability to lift and move heavy objects.Ability to follow detailed or general directions on how to perform tasks.Ability to work varying and unusual work schedules.Ability to follow and comply with employer's safety rules and regulations.CompanyUniversity of Illinois SpringfieldUniversity of Illinois Springfield, one of three universities in the world-class U of I system, is known for educating public servants and leaders.Founded in 1969Springfield, Illinois, USA501-1000 employeeshttp://www.uis.edu/FundingCurrent StageLate StageLeadership TeamTulio LlosaCIORecent NewsGovernment Technology USIllinois Universities Expand Online Education With Risepoint2025-06-28SlashGear6 Myths About SUVs You Need To Stop Believing2024-11-24StartuptoEnterpriseBreakthrough in Type 1 Diabetes Got iLet Bionic Pancreas System2022-05-14Company data provided by crunchbaseBoost Your Interview ChancesImprove Resume Match ScoreFREE?Your Score8.6Top ApplicantsMust-Have Skills for This RoleTheatrical lighting operationSound system operationStage riggingCostume maintenancePhysical staminaOptimize my ResumeGet Referral Via linkedIn FREE3× Higher Response via Email OutreachLLinda L.Technical Recruiter  \n\"\"\"\n\nSCHEMA:\n\"\"\"\n{\"$schema\":\"http://json-schema.org/draft-04/schema#\",\"type\":\"object\",\"properties\":{\"personal_information\":{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"email\":{\"type\":\"string\"},\"phone\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"socials\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"link\":{\"type\":\"string\"}},\"required\":[\"name\",\"link\"]}]}},\"required\":[\"name\",\"email\",\"phone\",\"location\"]},\"summary\":{\"type\":\"string\"},\"experiences\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"designation\":{\"type\":\"string\"},\"companyName\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"points\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"designation\",\"companyName\",\"location\",\"start_date\"]}]},\"education\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"institution\":{\"type\":\"string\"},\"degree\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"gpa\":{\"type\":\"string\"}},\"required\":[\"institution\",\"degree\",\"location\",\"start_date\",\"gpa\"]}]},\"skills\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"data\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"name\",\"data\"]}]},\"projects\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"projectName\":{\"type\":\"string\"},\"caption\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"url\":{\"type\":\"string\"},\"projectDetails\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]},\"externalSources\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"link\":{\"type\":\"string\"}},\"required\":[\"name\",\"link\"]}]},\"technologiesUsed\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"projectName\",\"location\",\"projectDetails\"]}]},\"certifications\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"issuing_organization\":{\"type\":\"string\"},\"issue_date\":{\"type\":\"string\"},\"expiration_date\":{\"type\":\"string\"},\"credential_id\":{\"type\":\"string\"},\"url\":{\"type\":\"string\"}},\"required\":[\"name\",\"issuing_organization\",\"issue_date\",\"expiration_date\",\"credential_id\",\"url\"]}]},\"awards\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"type\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"date\":{\"type\":\"string\"},\"description\":{\"type\":\"string\"}},\"required\":[\"name\",\"type\",\"location\",\"date\",\"description\"]}]},\"extracurricular_achievements\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"type\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"date\":{\"type\":\"string\"},\"description\":{\"type\":\"string\"}},\"required\":[\"name\",\"type\",\"location\",\"date\",\"description\"]}]},\"languages\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"language\":{\"type\":\"string\"},\"proficiency\":{\"type\":\"string\"}},\"required\":[\"language\",\"proficiency\"]}]}},\"required\":[\"personal_information\",\"education\",\"skills\",\"extracurricular_achievements\"]}\n\"\"\"\n\nCreate a tailored resume in JSON following the SCHEMA exactly.\nUse only content from the RESUME_TEXT but rewrite it to match the JOB_DESCRIPTION.\nDo not add extra lines or explanation.\nOutput only JSON.\n"},
]

# Apply chat template and tokenize properly
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# Tokenize the text
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=4096,
    do_sample=False
)

# Get only the generated tokens (exclude input)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# Decode the output
content = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

print("Generated Resume:")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

with open(f'generated_resume_{timestamp}.txt', 'w', encoding='utf-8') as f:
    f.write(content)

print("\nContent saved to 'generated_resume.txt'")

In [ ]:
# Save generated content
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f'./results/basemodelgenerated_resume_base_{timestamp}.txt'

with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(content)

print(f"✅ Base model output saved to: {output_filename}")

---

## 9. 🎯 LoRA Fine-tuning

Fine-tune the Qwen3 model using **QLoRA** (Quantized Low-Rank Adaptation) technique.

### Training Configuration:
- **Method:** QLoRA (4-bit quantization + LoRA adapters)
- **LoRA Rank (r):** 16
- **LoRA Alpha:** 32
- **Target Modules:** q_proj, k_proj, v_proj, o_proj
- **Epochs:** 2
- **Batch Size:** 1 (with gradient accumulation = 8)
- **Learning Rate:** 2e-4

### 9.1 Import Training Libraries

In [ ]:
# Import training-specific libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
import importlib.util

print("✅ Training libraries imported!")

### 9.2 Load Model for Training

In [ ]:
# Load model with quantization for training
model_name = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"🔄 Loading model for training: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print("✅ Model loaded for training!")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.00s/it]



### 9.3 Prepare Dataset

In [ ]:
# Prepare dataset with chat template
def format_chat(example):
    """Apply chat template to convert messages to training text."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )
    return {"text": text}

# Load and format dataset
dataset = load_dataset("json", data_files={"train": "dataset/training/final_training_dataset.jsonl"})
train_dataset = dataset["train"].map(format_chat)

print(f"📊 Training samples: {len(train_dataset)}")

### 9.4 Configure LoRA

In [ ]:
# Configure LoRA adapters
lora_config = LoraConfig(
    r=16,                                           # LoRA rank
    lora_alpha=32,                                  # LoRA alpha (scaling factor)
    lora_dropout=0.05,                              # Dropout for regularization
    bias="none",                                    # Don't train bias terms
    task_type="CAUSAL_LM",                          # Causal language modeling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]  # Attention layers
)

print("✅ LoRA configuration:")
print(f"   Rank: {lora_config.r}")
print(f"   Alpha: {lora_config.lora_alpha}")
print(f"   Dropout: {lora_config.lora_dropout}")
print(f"   Target modules: {lora_config.target_modules}")

### 9.5 Configure Training Arguments

In [ ]:
# Configure SFT training
sft_config = SFTConfig(
    output_dir="./qwen3-resume-lora",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,    # Effective batch size = 8
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=500,
    bf16=True,                        # Use bfloat16 precision
    warmup_steps=50,
    dataset_text_field="text",
    packing=False,                    # Keep full conversations
)

print("✅ Training configuration:")
print(f"   Epochs: {sft_config.num_train_epochs}")
print(f"   Batch size: {sft_config.per_device_train_batch_size}")
print(f"   Gradient accumulation: {sft_config.gradient_accumulation_steps}")
print(f"   Learning rate: {sft_config.learning_rate}")

### 9.6 Initialize Trainer and Train

In [ ]:
# Initialize SFT Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    peft_config=lora_config,
    args=sft_config,
)

print("✅ Trainer initialized!")

Truncating train dataset: 100%|██████████| 1530/1530 [00:00<00:00, 18434.17 examples/s]



In [ ]:
# Start training
print("🚀 Starting training...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
c:\ProgramData\anaconda3\envs\finetune\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\ProgramData\anaconda3\envs\finetune\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentra

Step,Training Loss
20,3.622900
40,3.181300
60,2.981000
80,2.978000
100,2.932100
120,2.914000
140,2.931200
160,2.924000
180,2.927900
200,2.931700


TrainOutput(global_step=384, training_loss=2.939210648338, metrics={'train_runtime': 4006.2222, 'train_samples_per_second': 0.764, 'train_steps_per_second': 0.096, 'total_flos': 6.853413121818624e+16, 'train_loss': 2.939210648338, 'entropy': 2.7159252166748047, 'num_tokens': 3133440.0, 'mean_token_accuracy': 0.4857884015028293, 'epoch': 2.0})

In [ ]:
# Save the trained model
trainer.save_model("./qwen3-resume-lora")
tokenizer.save_pretrained("./qwen3-resume-lora")

print("✅ Training completed!")
print("📁 Model saved to: ./qwen3-resume-lora")

Training completed!


---

## 10. 🚀 Inference with Fine-tuned Model

Load the fine-tuned LoRA adapter and generate tailored resumes.

### 10.1 Load Fine-tuned Model

In [ ]:
# Load fine-tuned model for inference
model_name = "Qwen/Qwen3-4B-Instruct-2507"
lora_adapter_path = "./qwen3-resume-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Check Flash Attention
flash_attn_available = importlib.util.find_spec("flash_attn") is not None
use_flash_attn = (flash_attn_available and torch.cuda.is_available() and 
                  torch.cuda.get_device_properties(0).major >= 8)

print(f"🔄 Loading base model: {model_name}")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2" if use_flash_attn else "eager",
    torch_dtype=torch.bfloat16,
)

# Load LoRA adapter
print(f"🔄 Loading LoRA adapter from: {lora_adapter_path}")
inference_model = PeftModel.from_pretrained(base_model, lora_adapter_path)

# Clean up
del base_model
torch.cuda.empty_cache()
import gc
gc.collect()

# Load tokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(lora_adapter_path)

print("✅ Fine-tuned model loaded successfully!")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.03s/it]



Model and tokenizer loaded successfully!


### 10.2 Generate Tailored Resume

In [ ]:
# Test with your sample messages
test_messages =  [
    {"role": "system", "content": "You create a tailored resume based on the job description.\n\nYour task:\n1. Read the RESUME_TEXT.\n2. Read the JOB_DESCRIPTION.\n3. Use only the information inside these two.\n4. Follow the SCHEMA exactly.\n5. Write a tailored resume in JSON using the SCHEMA.\n6. Do not output anything outside the JSON.\n7. If a field is missing in the resume, write a short, safe placeholder that fits the job.\n\n"},
    {"role": "user", "content": "\nRESUME_TEXT:\n\"\"\"\nPHANI SETTY 214 9235723 settyphanigmailcom Summary A handson programmer in userinterface design and development with over 20 years of experience and a proven track record in shipping web experiences for singlepage applications hybrid apps online stores and interacting market content Extensive experience in Architecting and developing for large and distributed frontend codebases as well as restructuring legacy systems to reduce bloat increase modularity and speed up future development iterations I have worked with a variety of web application stacks along with their respective view and templating systems When I am not coding I am involved in UXside of things like sketching out wireframes and designs on paper or whiteboard creating highfidelity designs and clickable prototypes and usertesting them I have developed innovative products and express brands through strategically driven design and interactive models 15 years of handson experience in leading teams building highperforming teams of onsite remote and offshore developers and designers Afterhours Im an avid learner of new technologies through personal projects as well as writing a book I was a Certified PMP20062009 and a Certified ScrumMasterpresent I bring people and technology together through effective communication and leadership Im equally comfortable discussing business requirements with product managers architects and APIs with developers Ive designed developed and delivered technology solutions in both traditional Waterfall and Agile development environments as well as transitioning between them I Have worked extensively in Healthcare Education Management Consulting domains I have strong technical and communication skills Core Competencies CSS Architecture I Frontend Architecture I Design Systems I Adaptive responsive design I wireframing Prototyping browser device debugging I Pageload Performance tuning I Mentoring and Training I Agile software development Technical Competencies Clientside UI JavaScript JQuery ReactJS redux Angular Websockets RequireJS HeatmapJS TypeScript HTMS CSS pre post processing SVG Backbone Handlebars D3 Environments ASPNET Core Web Forms and MVC Nodejs Java Grails PHP Git Github Gitlab CVS TFS Tools Visual Studio Code IntelliJ Adobe CC suite Education BE Computer Science Amravati University 1996 BSIT from Grantham University2017 MSSoftware Engineering from Walden University2021 Certifications Certified scrum master2018 Certified PMP2006 Six sigmagreen belt 2002 Projects CBRE UI Lead July 2018 Till Date Dallas tx Responsibilities As the onshore lead for a large consulting team I oversaw coding standards related to Angular 28 CSS JavaScript I wrote prototypes as well as augmented teams to facilitate project completion My UI organization included creating and administering training programs and overseeing code reviews for a global team of developers at various skill levels along with working closely with clients being empathetic to their needs all the while interpreting them in a cognitive experience that makes sense to their customer Accomplished Web project objectives by establishing clear understanding of project requirements Involved in designing and developing the components using HTML CSS JavaScript Bootstrap SASS Angular8 Flex and NodeJS Involved in implementing various screens for the frontend using Angular and used various public libraries from NPM Node Package Manager Collaborate crossfunctionally to develop research plan user flows information architecture and wireframes and work with Agile Scrum teams Managed a growing team of UI Engineers focused on building great experiences teaching and implementing excellence Visualize large data and develop dashboards As a lead Have good ability to analyze problems find solutions and implement them to tight deadlines on time I have good experience writing technical briefs technical specifications and generating costingtimings for projects Dealer socket Tech Leadui apr2018 July 2018 irving tx Responsibilities As a Lead I am responsible for making sure a quality and a welldocumented and test driven code is developed Extensive experience in building many software solutions with particular emphasis on clientside code Web Apps and Native Mobile Apps Specialized in architecting UI frameworks and creating custom reusable user interface components Create responsive landing pages and email templates for product communications Create and manage paid social media ad campaigns to drive lead generation Develop maintain systems utilizing HTML CSS JavaScriptjQuery Develop maintain systems utilizing client side frameworks and libraries Work closely with product owners backend C developers and UX designers to implement new features Mobile Applications Web Single Page Applications UI Components development WebComponents React Redux ReduxSaga ES Flow Jest Enzyme Babel NodeJS Webpack CI with Bitbuckets Pipeline and Docker containers HarMAN internationalTech leadui Dec2017 April 2018Plano tx Responsibilities As a Lead I am responsible for making sure a quality and a welldocumented code is developed Prototypingarchitecting and implementation of UI Secure clustering components and pages using ReactRedux Bootstrap UI RESTfetch Knex Bookshelf SASS GitGithub Webpack CreatingRefactoring ReactRedux reusable components for integration into encrypting dependencies Design and develop reusable Angular 4 Node Package Modules Provide guidance on technical architecture using HTML5JavaScript web technologies Builds responsive web user interfaces that ensure seamless user experience across desktop and mobile platforms Builds fullduplex embedded web applications that control hardware devices in realtime using web sockets Establishes UI development process for embedded devices Unit testing with all elements of ReactRedux project by using Jest Enzyme Adjustment desktop web apps for mobilesize devices such as smartphones and tablets Mock servers coding on Python and Golang for unit testing on local environment Integration testing with Selenium IBMUI ArchitectJuly 2017 dec2017 Santa Clara CA Responsibilities DesignArchitect support and lead both offshore and onsite teamsabout 10 React and AngularUI developers as part of refactoring an IBMs flagship product Worked with modules like MongoDB and mongoose for database persistence using Nodejs to interact with mongodb Worked with unit testing of javascript applications using Karma Jasmine apimocker Jest enzyme snion Tasks include making sure a quality and a welldocumented code is developed and ESLint errors are fixed unit tests are written and the CI builds are through for code merge Interpret clients needs and ability to architect design and develop solutions with high visual impact to get the clients message across I am responsible for user interface strategy planning development and delivery across the practice My duties are to maintain consistent design guidelines best practices and standards and project methodologies My role as the Architect for UI is to work with product owners and stakeholders to present and create solutions to visualize design and deliver style guides prototypes and assets for the client with full compliance of client rules and guidelines Optum technologiesuhg dev LEAD April2016 May 2017 Minneapolis Plano Responsibilities Evaluate and implement SEO friendly infrastructures in HTML and Javascript to new and existing applications Migrating legacy Angular 14 components to higher versions at the moment Front end architecture and development of largescale Angular application in the AEM environment Developing a scaffolding system to build the frontend using Gulp along with integrating SASS preprocessing and Bootstrap library SPAs using AngularJS 1x including factory services and directives consumption of web services Consulting on UX and visual design options Recommending design and technical solutions based on user requirements and business needs Develop poling and cross team communication ApplicationPCTC Developed documentation testing standards Implementation of Bootstrap Foundation and Jquery frameworks HealthPartners Senior Web DeveloperUI Lead Jan 2016 March 2016 Minneapolis HealthPartners is an integrated nonprofit health care provider and health insurance company located in Bloomington Minnesota offering care coverage research and education to its members patients and the community Duties include working collaboratively with the team and management on designing and delivering complex crossbrowser applications and maintaining existing products Responsibilities I was programming Widgets and Applications for wwwHealthPartnerscom using Angularjs Javascript and BootStrap Design development and testing phases of Software Development using AGILE Methodology and Test Driven Development TDD Involvement in all stages of Software development life cycle including Analysis development Implementation testing and support Involved in development of User Interface using HTML5 CSS3 JavaScript and jQuery AJAX JSON Developed single page web application using JavaScript framework Worked with CrossBrowser Compatible issues Created reusable templates and style sheets based on UI standards and guidelines Performed Functional tasks using specifications and wireframes Extensively used Debugging Cascading Style Sheets to change the styles now and in the future Designed and implemented the UI with extensive use of JavaScript JSON and Ajax Designed and developed basic user interfaces to HealthPartners web services by analyzing business requirements and priorities Provides code and web design reviews Integrated applications to web services via server scripting and database architecting Troubleshoots development and production incidents across multiple environments and operating platforms Medtronic UI Lead Nov 2014 Dec 2015 Minneapolis Responsibilities I was responsible for Architecting used combination of Event MVC and AMD patterns designing and programming the frontend for an Electro Cradio Gram waveforms rendering mobile application using Javascript and HTML 5 canvas I have developed a handy UI widget library using pure Javascript for our internal app developers to create UI elements dynamically Managed software development operational and client integration projects Created a complete test bed for the UI usingstubbing the server using Node JS I have designed the widget library in the AMD pattern using RequireJS I have also developed various reports and charts using HTML Canvas HTML SVG D3JS and SVGjs by passing JSON objects or Arrays as input both for mobile and web applications I have worked extensively on Ajax and JavaScript Websockets I have revamped an existing single thread application that had heavy computational data in the UI to a light weight application using Web workers Have been working on Jasmine and Chutzpah for my unit testing as we develop using TDD approach in Visual StudioXamarin environment Worked extensively with jQuery HTML4 and CSS Developed prototypes and mokups using Adobe Fireworks Edge and Balsamiq Developed user friendly and attractive UI Have also hand coded web templates using HTML5 Javascript Bootstrap and CSS3 Harvard Business Publishing Global engagement manager UI Lead Sep 2010 October 2014 Boston MA Responsibilities Managing the global vendors offering services in content development and translationlocalization Gathering and analyzing requirements and writing SOWs and scope documents along with leading the project from concept to completion Responsible for User Interface design and development for Harvard Business Publishings portals and products like LeadershipDirectorg WCMS and Harvard Manage Mentor using Adobe Photoshop CS5 Adobe Flash CS5 Illustrator HTML5 Javascript Angular JS JQuery CSS3 and AJAX I was responsible for designing solutions that improved user experience and supported graphic resource needs for various products for Harvard Business Publishing Boston MA I was also responsible for creatinguser personas creating wireframes visual mockups UI specs and flash designs for service window applications Created design strategy and implemented in various UIUX projects I have conducted research and data evaluation on interactive products and created graphics animations using flash and after effects for various service window applications Responsible for reviewing creative work provide art direction and design feedback when working with junior designers and developers My technical and creative Skills helped me to work on multiple projects simultaneously I have worked with product management and engineering teams to ensure that the graphics and layout designs meet customer requirements and implementation constraints GE Sr UI Developerproject managerTech lead Oct 2004 Sep 2010 Albany New York Responsibilities Gathering and analyzing requirements and writing SOWs and scope documents along with leading the project from concept to completion Responsible for User Interface Design for web applications and learning portals using Adobe Photoshop CS5 Adobe Flash CS5 Illustrator HTML5 Javascript CSS3 AJAXand the clients preferred content management tools like Knet Participate in early sprints of agile methods Work ahead of sprint and keep the work ready for the development team for development Created the rich user experience and user interface design for their Archeological data collection application for constructions process Involved in the hiring process and recruited worldclass talent user interface development group Support flash action script for interactive service window applications across the apps team Involved in the research and discover phase of the apps development for client and user research Created the rich presentations for the corporate initiatives United Nations Development Program UI Dev Jul 2002 Sep 2004 New York Responsibilities Responsible for User Interface Design using Adobe Photoshop Flash HTML CSS JavaScript and the companys custom web development and content management tools Involve in daily scrums create IA Visual design for agile process Supported interaction design visual design for web products and produced images banner logo poster brochure using softwares like Photoshop Image ready Illustrator In Design IMI Web designer Jan 1999 Jun 2002 Hyderabad India Responsibilities Responsible for User Interface Design and development for various mobilewebdesktop applications for Vodafone I have also created the interface for WAP based application called VOOP Virtual Office on Phone the application enables the users to remotely access their PCs from mobiles to send mails edit documents etc I have also produced entire UI kit for transmission tower designing applications using Photoshop and icon maker tools This was challenging back then as these tools were coded in VC and VC supported transparency only in 256 color ico formats Fountainhead Design Studios Graphic Designer Jan 1997 Jan 1999 Hyderabad India Responsibilities I have created 120 animated greeting cards for Archies online portal using Macromedia Flash 40 on iMac I was involved in the process right from conceptualizing to completion of the greeting cards I have also produced about 150 printed greeting cards using Photoshop Bryce 3D and Corel Draw I have also designed and developed more than 50 websites using basic HTML Macromedia Flash 40 and Dreamweaver 20\n\"\"\"\n\nJOB_DESCRIPTION:\n\"\"\"\nExtra Help Stagehand - 50 vacancies jobs in United StatesOverviewCompanyThis job has closed.APPLY to similar jobsUniversity of Illinois Springfield · 4 weeks agoExtra Help Stagehand - 50 vacanciesSpringfield, ILFull-timeOnsiteEntry Level$20.07/hr - $20.07/hrThe University of Illinois Springfield is seeking Extra Help Stagehands to support their theatrical productions. The role involves coordinating and executing various tasks related to the preparation, operation, and maintenance of event equipment, stage lighting, sound systems, and theatrical scenery, ensuring a safe and efficient working environment.EducationUniversitiesResponsibilitiesAttendance (paid) at employer-required safety training sessionsLoad & unload theatrical equipment into and out of vehicles as required including stacking and unstacking of equipmentMove theatrical equipment into and out of venue storage areasPerform all work required to operate and maintain the safety, trim, balance, and proper rigging of the counterbalanced fly system and associated winched cables, pin-rail equipment and pick-linesPerform such technical specialties as the splicing of cables and ropes, the use of stage weights and braces, the maintenance and correct use of tie lines and pick lines, the maintenance and repair of curtains, and the periodic inspection of curtains in storage to prevent damage.Operate mechanical systems such as pit-lifts, orchestra shell winches, chain motors, etc. Operating of chain motors does NOT also include functions performed by EXTRA HELP RIGGERS.Operate personnel-lift devicesHang, connect to the dimming system, lamp, focus and color theatrical lighting instruments for general and specific usesMove and place audio gear such as microphones, speakers, monitors, control boards and cable as requiredInstall, remove, operate and reconfigure stage curtains, portable flooring, scenery, props and other theatrical items as required.Sweeping and mopping of stage area as required for safety and proper audience presentationMaintain, move, set up and operate spotlights as requiredMaintain and operate lighting and audio control systemsInstall, maintain and operate as needed equipment hung from counter-weighted battens.Setup, maintain (including laundry), repair and place back into storage costumes, wigs and other items worn by performers including towels and associated fabric items. Also assist performers with putting on and taking off costume items.Perform other duties commonly associated with stage operations as requiredQualificationTheatrical lighting operationSound system operationStage riggingCostume maintenancePhysical staminaSafety complianceTeam collaborationAttention to detailRequiredBasic entry-level knowledge of a back-stage theatrical working environment including terminology & standard operating procedures usually acquired through formal education in the performing arts field or previous volunteer/work experience as a stage crew member in a high school, college, community theatre, music club, regional theatre, or professional performance venue.Normal ability to hear, see, speak, smell in a backstage environment which at times can be darkened, crowded, and noisy with multiple hazards.Ability to move quickly unaided.Ability to climb stairs, ladders and work at heights.Ability to lift and move heavy objects.Ability to follow detailed or general directions on how to perform tasks.Ability to work varying and unusual work schedules.Ability to follow and comply with employer's safety rules and regulations.CompanyUniversity of Illinois SpringfieldUniversity of Illinois Springfield, one of three universities in the world-class U of I system, is known for educating public servants and leaders.Founded in 1969Springfield, Illinois, USA501-1000 employeeshttp://www.uis.edu/FundingCurrent StageLate StageLeadership TeamTulio LlosaCIORecent NewsGovernment Technology USIllinois Universities Expand Online Education With Risepoint2025-06-28SlashGear6 Myths About SUVs You Need To Stop Believing2024-11-24StartuptoEnterpriseBreakthrough in Type 1 Diabetes Got iLet Bionic Pancreas System2022-05-14Company data provided by crunchbaseBoost Your Interview ChancesImprove Resume Match ScoreFREE?Your Score8.6Top ApplicantsMust-Have Skills for This RoleTheatrical lighting operationSound system operationStage riggingCostume maintenancePhysical staminaOptimize my ResumeGet Referral Via linkedIn FREE3× Higher Response via Email OutreachLLinda L.Technical Recruiter  \n\"\"\"\n\nSCHEMA:\n\"\"\"\n{\"$schema\":\"http://json-schema.org/draft-04/schema#\",\"type\":\"object\",\"properties\":{\"personal_information\":{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"email\":{\"type\":\"string\"},\"phone\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"socials\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"link\":{\"type\":\"string\"}},\"required\":[\"name\",\"link\"]}]}},\"required\":[\"name\",\"email\",\"phone\",\"location\"]},\"summary\":{\"type\":\"string\"},\"experiences\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"designation\":{\"type\":\"string\"},\"companyName\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"points\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"designation\",\"companyName\",\"location\",\"start_date\"]}]},\"education\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"institution\":{\"type\":\"string\"},\"degree\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"gpa\":{\"type\":\"string\"}},\"required\":[\"institution\",\"degree\",\"location\",\"start_date\",\"gpa\"]}]},\"skills\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"data\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"name\",\"data\"]}]},\"projects\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"projectName\":{\"type\":\"string\"},\"caption\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"url\":{\"type\":\"string\"},\"projectDetails\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]},\"externalSources\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"link\":{\"type\":\"string\"}},\"required\":[\"name\",\"link\"]}]},\"technologiesUsed\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"projectName\",\"location\",\"projectDetails\"]}]},\"certifications\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"issuing_organization\":{\"type\":\"string\"},\"issue_date\":{\"type\":\"string\"},\"expiration_date\":{\"type\":\"string\"},\"credential_id\":{\"type\":\"string\"},\"url\":{\"type\":\"string\"}},\"required\":[\"name\",\"issuing_organization\",\"issue_date\",\"expiration_date\",\"credential_id\",\"url\"]}]},\"awards\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"type\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"date\":{\"type\":\"string\"},\"description\":{\"type\":\"string\"}},\"required\":[\"name\",\"type\",\"location\",\"date\",\"description\"]}]},\"extracurricular_achievements\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"type\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"date\":{\"type\":\"string\"},\"description\":{\"type\":\"string\"}},\"required\":[\"name\",\"type\",\"location\",\"date\",\"description\"]}]},\"languages\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"language\":{\"type\":\"string\"},\"proficiency\":{\"type\":\"string\"}},\"required\":[\"language\",\"proficiency\"]}]}},\"required\":[\"personal_information\",\"education\",\"skills\",\"extracurricular_achievements\"]}\n\"\"\"\n\nCreate a tailored resume in JSON following the SCHEMA exactly.\nUse only content from the RESUME_TEXT but rewrite it to match the JOB_DESCRIPTION.\nDo not add extra lines or explanation.\nOutput only JSON.\n"},
]


prompt = inference_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)
    
# Tokenize input
inputs = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)

# Generate response
with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=4096,
        do_sample=False,
        # temperature=0.2,
        # top_p=0.9,
        pad_token_id=inference_tokenizer.eos_token_id,
    )

# Decode only the generated part (exclude the prompt)
generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
response = inference_tokenizer.decode(generated_tokens, skip_special_tokens=True)
    

print("Generated Tailored Resume:")
print("-" * 50)
print(response)

output_filename = f"./results/finetuned/finetuned_generated_resume_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(response)

print(f"Resume saved to: {output_filename}")

# Try to parse as JSON to validate
try:
    resume_json = json.loads(response)
    print("\n✅ Valid JSON generated!")
    print(json.dumps(resume_json, indent=2)[:2000])  # Print first 2000 chars
except json.JSONDecodeError as e:
    print(f"\n⚠️ Response is not valid JSON: {e}")
    print("Raw response saved to file.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated Tailored Resume:
--------------------------------------------------
{
  "personal_information": {
    "name": "PHANI SETTY",
    "email": "settyphanigmailcom",
    "phone": "214 9235723",
    "location": "USA",
    "socials": []
  },
  "summary": "A handson programmer in userinterface design and development with over 20 years of experience and a proven track record in shipping web experiences for singlepage applications hybrid apps online stores and interacting market content Extensive experience in Architecting and developing for large and distributed frontend codebases as well as restructuring legacy systems to reduce bloat increase modularity and speed up future development iterations I have worked with a variety of web application stacks along with their respective view and templating systems When I am not coding I am involved in UXside of things like sketching out wireframes and designs on paper or whiteboard creating highfidelity designs and clickable prototypes and use

In [ ]:
# Save and validate generated resume
output_filename = f"./results/finetuned/finetuned_generated_resume_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(response)

print(f"📄 Resume saved to: {output_filename}")

# Validate JSON format
try:
    resume_json = json.loads(response)
    print("\n✅ Valid JSON generated!")
    print(json.dumps(resume_json, indent=2)[:2000] + "...")
except json.JSONDecodeError as e:
    print(f"\n⚠️ Response is not valid JSON: {e}")
    print("Raw response saved to file.")

### 10.3 Clean Up GPU Memory

In [ ]:
# Clean up GPU memory
import gc

# Delete model objects
for var in ['trainer', 'model', 'base_model', 'inference_model']:
    if var in dir():
        exec(f"del {var}")

# Clear Python garbage collector
gc.collect()

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
    print(f"💾 GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"💾 GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    print("✅ GPU memory cleaned!")
else:
    print("No CUDA device available")

GPU Memory Allocated: 0.00 GB
GPU Memory Reserved: 0.00 GB
✅ GPU memory cleaned!


---

## 11. 🔄 Alternative Training (Single GPU Stable) 

### Finally we are using this because it is far more stable and gave us the best result

This section provides a more stable training configuration for single-GPU setups.

### Key Differences:
- Uses `fp16` instead of `bf16` for better compatibility
- Includes `prepare_model_for_kbit_training` for QLoRA stability
- Pre-tokenizes and truncates dataset

In [ ]:
#!/usr/bin/env python
"""
Single-GPU QLoRA fine-tuning for Qwen3-4B-Instruct on resume->JSON data.
- Uses 4-bit quantization (bitsandbytes)
- Uses TRL SFTTrainer
- Assumes dataset JSONL with {"messages": [ ... ]} per row
"""

import os

# -------------------------------------------------------------
# 0. LOCK TO A SINGLE GPU (VERY IMPORTANT)
# -------------------------------------------------------------
# If you run this as a script (`python train_qwen3_resume.py`),
# this will ensure only GPU 0 is visible → no DataParallel,
# no CUBLAS / illegal memory access issues.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

# -------------------------------------------------------------
# 1. BASIC PATHS / NAMES
# -------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
DATA_PATH = "./final_training_dataset_cleaned.jsonl"  # your train.jsonl
OUTPUT_DIR = "./qwen3-resume-lora-single-gpu"
MAX_SEQ_LENGTH = 4096

# -------------------------------------------------------------
# 2. BITSANDBYTES 4-BIT QUANT CONFIG (STABLE ON 3090)
# -------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16 is safer than bf16 here
    bnb_4bit_use_double_quant=True,
)

# -------------------------------------------------------------
# 3. TOKENIZER
# -------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Ensure pad token is set (needed for Trainer)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Make sure long prompts are allowed
tokenizer.model_max_length = MAX_SEQ_LENGTH
tokenizer.init_kwargs["model_max_length"] = MAX_SEQ_LENGTH

# -------------------------------------------------------------
# 4. BASE MODEL (4-BIT, SINGLE GPU)
# -------------------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",   # with CUDA_VISIBLE_DEVICES=0 this will pick the only GPU
)

# Prepare model for k-bit training (IMPORTANT for QLoRA)
model = prepare_model_for_kbit_training(model)

# -------------------------------------------------------------
# 5. DATASET: CONVERT messages -> flat "text" USING QWEN CHAT TEMPLATE
# -------------------------------------------------------------
def format_chat(example):
    """
    Expects example like:
    {
      "messages": [
        {"role": "system", "content": "..."},
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "{...JSON...}"}
      ]
    }
    Produces one long training text string.
    """
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


raw_dataset = load_dataset("json", data_files={"train": DATA_PATH})
train_dataset = raw_dataset["train"].map(
    format_chat,
    remove_columns=raw_dataset["train"].column_names,  # keep only "text"
)

# Pre-tokenize and truncate the dataset
def tokenize_and_truncate(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    return tokens

train_dataset = train_dataset.map(
    tokenize_and_truncate,
    remove_columns=["text"],
)

# -------------------------------------------------------------
# 6. LoRA CONFIG (ATTN PROJECTIONS ONLY)
# -------------------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Qwen-safe target modules
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# -------------------------------------------------------------
# 7. SFT CONFIG (MATCHES YOUR PREVIOUS WORKING STYLE)
# -------------------------------------------------------------
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=500,
    warmup_steps=50,
    fp16=True,                  # Enable fp16 training
    bf16=False,                 # Disable bf16
    packing=False,              # keep full conversations, no packing
    remove_unused_columns=True,
    report_to="none",
)

# -------------------------------------------------------------
# 8. DATA COLLATOR
# -------------------------------------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

# -------------------------------------------------------------
# 9. TRAINER
# -------------------------------------------------------------
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    peft_config=lora_config,
    args=sft_config,
    data_collator=data_collator,
)

# -------------------------------------------------------------
# 10. TRAIN + SAVE
# -------------------------------------------------------------
if __name__ == "__main__":
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"✅ Training completed. LoRA adapter saved to: {OUTPUT_DIR}")

Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.45s/it]

Map: 100%|██████████| 1530/1530 [00:18<00:00, 84.41 examples/s]

Truncating train dataset: 100%|██████████| 1530/1530 [00:00<00:00, 177322.05 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/home/har5ha/miniconda3/envs/finetune/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling che

Step,Training Loss
20,3.611000
40,3.181200
60,2.977200
80,2.975000
100,2.928500
120,2.912200
140,2.929700
160,2.922500
180,2.926500
200,2.932300


✅ Training completed. LoRA adapter saved to: ./qwen3-resume-lora-single-gpu


In [ ]:
# Sample test messages for inference
# Note: This is a truncated example - full resume and job description would be much longer
test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "RESUME_TEXT: [Your resume text here]\n\nJOB_DESCRIPTION: [Job description here]\n\nSCHEMA: [JSON schema]"},
]

print("📝 Test messages configured for inference")

# 12 Load and Run Merged Model

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel

# Load and merge LoRA adapter for faster inference
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
LORA_PATH = "./qwen3-resume-lora-single-gpu"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Check Flash Attention
flash_attn_available = importlib.util.find_spec("flash_attn") is not None
use_flash_attn = (flash_attn_available and torch.cuda.is_available() and 
                  torch.cuda.get_device_properties(0).major >= 8)

print(f"🔄 Loading model with LoRA from: {LORA_PATH}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2" if use_flash_attn else "eager",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load and merge LoRA
model = PeftModel.from_pretrained(model, LORA_PATH)
model = model.merge_and_unload()  # Merge for faster inference
model.eval()

print("✅ Model loaded and merged!")

def generate(messages):
    """Generate response from the fine-tuned model."""
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.0,  # Deterministic for JSON
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    output_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return output_text

# Generate and save
output = generate(test_messages)
output_filename = f"./results/finetuned/finetuned_generated_resume_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(output)

print(f"✅ Generated resume saved to: {output_filename}")

Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.03s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


---

## 📊 Summary

This notebook demonstrated a complete pipeline for building a resume optimization system:

### Key Achievements:
1. ✅ Extracted text from **1800+ resumes** in PDF, DOCX, and DOC formats
2. ✅ Built a **job scraper** to collect job descriptions from the web
3. ✅ Created a **balanced dataset** matching resumes with job descriptions
4. ✅ Generated **tailored resumes** using both Ollama (local) and Gemini (cloud) APIs
5. ✅ Prepared training data in **chat format** for fine-tuning
6. ✅ Fine-tuned **Qwen3-4B** using **QLoRA** technique
7. ✅ Deployed the fine-tuned model for **inference**

### Model Details:
- **Base Model:** Qwen/Qwen3-4B-Instruct-2507
- **Fine-tuning Method:** QLoRA (4-bit quantization + LoRA)
- **Training Data:** ~1500 resume-job pairs with tailored outputs
- **Output Format:** Structured JSON following a predefined schema

## 📈 Model Performance

### 📊 Quality Evaluation Results

The fine-tuned model was evaluated against multiple outputs using **GPT-5.1 Thinking LLM** 🔗<u>[Link](https://chatgpt.com/s/t_69309bb9c0a0819191373e1d9cbe86d9)</u> as an expert evaluator to assess quality improvements:

| Output | Source | Score | Interpretation |
|--------|--------|-------|----------------|
| **Output 4** | Finetuned with good parameters + correct training pattern | ⭐ **9.5/10** | Best model. Fine-tuning succeeded. |
| **Output 2** | Base model (no finetune) | ⭐ **9/10** | Strong baseline. |
| **Output 1** | Finetuned on bad dataset | ⭐ **7/10** | Fine-tuning made it *worse* because data was flawed. |
| **Output 3** | Old training data | ⭐ **2/10** | Very harmful training data; must never be used. |

### Detailed Output Analysis (GPT-5.1 Evaluation)

#### 🟩 Output 4 — Best (Score: 9.5/10)
**Verdict: The cleanest, safest, and most schema-consistent version**
- ✅ Uses strictly valid JSON
- ✅ No hallucinations
- ✅ Highly aligned to job description
- ✅ Strong action verbs
- ✅ Covers all major responsibilities
- ✅ Clean, consistent skill blocks

#### 🟩 Output 2 — Strong Baseline (Score: 9/10)
**Verdict: Excellent — Good reference standard**
- ✅ Highly aligned to job description
- ✅ JSON is valid
- ✅ No hallucinations
- ▢ Slightly more verbose than Output 4

#### 🟨 Output 1 — Decent (Score: 7/10)
**Verdict: Acceptable but can be improved**
- ✅ Valid JSON
- ✅ Tailored to job description
- ✅ No hallucinations
- ▢ Less comprehensive than Output 2 & 4
- ▢ Slightly generic phrasing

#### 🟥 Output 3 — Very Poor (Score: 2/10)
**Verdict: Harmful for finetuning — Should NEVER be used**
- ❌ Not valid JSON (illegal commas, structural breaks)
- ❌ Violates schema in multiple places
- ❌ Completely ignores prompt instructions
- ❌ Hallucinates companies, jobs, degrees, certifications
- ❌ Random and inconsistent placeholder styles

### Key Findings

**What makes a good training sample:**
- ✅ Valid JSON with strict schema adherence
- ✅ No hallucinations or invented data
- ✅ Closely aligned to job description
- ✅ Strong action verbs in experience descriptions
- ✅ Clean, consistent skill blocks
- ✅ Professional summary tone

**What to avoid in training data:**
- ❌ Invalid JSON (illegal commas, structural breaks)
- ❌ Schema violations
- ❌ Hallucinated companies, jobs, degrees, or certifications
- ❌ Ignoring prompt instructions
- ❌ Inconsistent placeholder styles
- ❌ Over-verbose or under-detailed descriptions

### Evaluation Criteria

1. **Schema Consistency** - Valid JSON, no hallucinations, proper structure
2. **Tailoring Strength** - Alignment with job description responsibilities
3. **Experience Quality** - Correct action verbs, within scope, no invented tasks
4. **Skill Block Quality** - Job-relevant, consistent structure, no fluff
5. **Summary Quality** - Crisp, accurate, professionally toned

### Key Metrics

- **Average Generation Time**: ~3-5 seconds per resume (on RTX 3090)
- **Max Sequence Length**: 4096 tokens
- **Output Format**: Deterministic JSON (temperature=0.0)

### Tips for Best Results

1. Use `temperature=0.0` and `do_sample=False` for consistent JSON output
2. Use `merge_and_unload()` for faster, more stable inference
3. Ensure the prompt follows the exact training format with RESUME_TEXT, JOB_DESCRIPTION, and SCHEMA sections
4. Use high-quality training data that follows schema strictly
5. Avoid training samples with hallucinations or schema violations



